In [2]:
import os, json, torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import re

PROJECT_ROOT  = r"D:\Pics Can Lie"
DATASET_ROOT  = os.path.join(PROJECT_ROOT, "dataset")
VAL_META_PATH = os.path.join(PROJECT_ROOT, "dataset", "data", "NewsClipPings", "metadata", "val.json")
WIKI_CACHE = r"E:\Pics Can Lie\wikipedia_text_cache.json"
DEBERTA_CACHE = os.path.join(PROJECT_ROOT, "deberta_val_scores_v2.csv")  # new file, keeps old
CLIP_IDS      = r"E:\Pics Can Lie\val_sample_ids.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load v2 ID mapping as master
id_df = pd.read_csv(CLIP_IDS)
id_df["id"] = id_df["id"].astype(str)
print(f"V2 samples to score: {len(id_df)}")

# Load metadata
with open(VAL_META_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

# Load Wikipedia cache
with open(WIKI_CACHE, "r", encoding="utf-8") as f:
    wiki_cache = json.load(f)
print(f"Wikipedia cache: {len(wiki_cache)} entries")

# Build sample list in v2 ID order
val_samples = []
for _, row in id_df.iterrows():
    art_id = str(row["id"])
    if art_id not in metadata:
        val_samples.append(None)
        continue
    meta     = metadata[art_id]
    entities = sorted(meta.get("caption_entities_rel", []),
                      key=lambda x: x[3], reverse=True)[:2]
    val_samples.append({
        "id":       art_id,
        "caption":  meta["caption"],
        "entities": entities,
        "label":    int(row["label"])
    })

print(f"Samples built: {sum(1 for s in val_samples if s is not None)} valid")

# Load DeBERTa
print("Loading DeBERTa...")
nli_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/nli-deberta-v3-large")
nli_model     = AutoModelForSequenceClassification.from_pretrained("cross-encoder/nli-deberta-v3-large")
nli_model.to(device)
nli_model.eval()
print("DeBERTa loaded.")

def get_entailment_score(caption, premise, max_len=512):
    if not premise or not caption:
        return 0.33
    try:
        inputs = nli_tokenizer(
            premise, caption,
            return_tensors="pt", truncation=True,
            max_length=max_len, padding=True
        ).to(device)
        with torch.no_grad():
            logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        return float(probs[2])
    except:
        return 0.33

# Score all samples
results = []
for sample in tqdm(val_samples, desc="DeBERTa NLI"):
    if sample is None:
        results.append({"id": None, "entailment_score": 0.33, "label": -1})
        continue

    # Build Wikipedia premise from cached text
    wiki_texts = []
    for ent in sample["entities"]:
        text = wiki_cache.get(ent[0], "")
        if text:
            wiki_texts.append(text[:500])
    premise = " ".join(wiki_texts)[:1000] if wiki_texts else sample["caption"]

    score = get_entailment_score(sample["caption"], premise)
    results.append({
        "id":               sample["id"],
        "entailment_score": score,
        "label":            sample["label"]
    })

# Save
df = pd.DataFrame(results)
df.to_csv(DEBERTA_CACHE, index=False)
scores = df["entailment_score"].values
print(f"\nSaved to {DEBERTA_CACHE}")
print(f"Scores: min={scores.min():.3f} max={scores.max():.3f} mean={scores.mean():.3f}")

Device: cuda
V2 samples to score: 5000
Wikipedia cache: 3243 entries
Samples built: 5000 valid
Loading DeBERTa...


Loading weights: 100%|██████████| 394/394 [00:00<00:00, 3231.69it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DeBERTa loaded.


DeBERTa NLI: 100%|██████████| 5000/5000 [09:04<00:00,  9.18it/s]


Saved to D:\Pics Can Lie\deberta_val_scores_v2.csv
Scores: min=0.000 max=1.000 mean=0.418


In [2]:
import pandas as pd
import numpy as np

deb = pd.read_csv(r'D:\Pics Can Lie\deberta_val_scores_v2.csv')

print('Score stats by label:')
real = deb[deb['label'] == 0]['entailment_score']
fake = deb[deb['label'] == 1]['entailment_score']

print(f'\n  entailment_score:')
print(f'    real : mean={real.mean():.4f}  std={real.std():.4f}  min={real.min():.4f}  max={real.max():.4f}')
print(f'    fake : mean={fake.mean():.4f}  std={fake.std():.4f}  min={fake.min():.4f}  max={fake.max():.4f}')
print(f'    diff : {fake.mean() - real.mean():+.4f}')

print()
print('Distribution (entailment_score buckets):')
bins = [0, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
print(f'  {"Bucket":<15} {"Real":>8} {"Fake":>8}')
for i in range(len(bins)-1):
    lo, hi = bins[i], bins[i+1]
    r = ((real >= lo) & (real < hi)).sum()
    f = ((fake >= lo) & (fake < hi)).sum()
    print(f'  {lo:.2f} – {hi:.2f}      {r:>8}  {f:>8}')

print()
print('Sample low vs high entailment examples:')
print('-- Low entailment (bottom 5):')
print(deb.nsmallest(5, 'entailment_score')[['id','entailment_score','label']].to_string())
print()
print('-- High entailment (top 5):')
print(deb.nlargest(5, 'entailment_score')[['id','entailment_score','label']].to_string())

Score stats by label:

  entailment_score:
    real : mean=0.4150  std=0.4773  min=0.0000  max=0.9999
    fake : mean=0.4216  std=0.4790  min=0.0000  max=0.9999
    diff : +0.0066

Distribution (entailment_score buckets):
  Bucket              Real     Fake
  0.00 – 0.01           474       464
  0.01 – 0.05           955       953
  0.05 – 0.10            12        11
  0.10 – 0.20            15        15
  0.20 – 0.50            21        16
  0.50 – 1.00          1023      1041

Sample low vs high entailment examples:
-- Low entailment (bottom 5):
           id  entailment_score  label
2905  1561698          0.000020      0
3298   957837          0.000029      1
3442   957837          0.000029      0
1135    89805          0.000036      1
45     139658          0.000037      1

-- High entailment (top 5):
           id  entailment_score  label
2513   725126          0.999924      0
2646  1119341          0.999924      1
4703  1119341          0.999924      0
2704  1683056          0

In [4]:
import json
import numpy as np

with open(r'D:\Pics Can Lie\kaggle_dataset_full\metadata\val.json', 'r') as f:
    metadata = json.load(f)

# Inspect entity fields on a few samples
print('=== Sample entity fields ===\n')
for sid, meta in list(metadata.items())[:5]:
    print(f'ID: {sid}')
    print(f'  caption              : {meta.get("caption","")[:100]}')
    print(f'  caption_entities_spacy: {meta.get("caption_entities_spacy")}')
    print(f'  caption_entities_rel : {meta.get("caption_entities_rel")}')
    print(f'  title_entities_spacy : {meta.get("title_entities_spacy")}')
    print(f'  title                : {meta.get("title","")[:100]}')
    print(f'  image_has_person     : {meta.get("image_has_person")}')
    print()

# Coverage stats
has_caption_ent = sum(1 for m in metadata.values() 
                      if m.get('caption_entities_spacy'))
has_rel_ent     = sum(1 for m in metadata.values() 
                      if m.get('caption_entities_rel'))
has_person      = sum(1 for m in metadata.values() 
                      if m.get('image_has_person'))

print(f'Samples with caption_entities_spacy : {has_caption_ent}/5000')
print(f'Samples with caption_entities_rel   : {has_rel_ent}/5000')
print(f'Samples with image_has_person       : {has_person}/5000')

=== Sample entity fields ===

ID: 931806
  caption              : People might assume that I m rolling in money Couldn t be further from the truth at the moment Will 
  caption_entities_spacy: [['Will Poulter', 'PERSON'], ['London', 'GPE'], ['Suki Dhanda', 'PERSON']]
  caption_entities_rel : [['Will_Poulter', 'PER', 'Will Poulter', 0.9183], ['London', 'LOC', 'London', 0.9982], ['The_New_York_Observer', 'ORG', 'Observer', 0.8858]]
  title_entities_spacy : [['Will Poulter', 'PERSON'], ['Soho House LA', 'FAC']]
  title                : Will Poulter: ‘Hanging out in Soho House LA, that’s my worst nightmare’
  image_has_person     : True

ID: 389177
  caption              : Garth Brooks performs during the Opening Ceremonies concert at the Lincoln Memorial for President Ba
  caption_entities_spacy: [['the Lincoln Memorial', 'FAC'], ['Garth Brooks', 'PERSON'], ['Barack Obama', 'PERSON'], ['the Opening Ceremonies', 'WORK_OF_ART']]
  caption_entities_rel : [['Garth_Brooks', 'PER', 'Garth Brook

In [5]:
import os, json

sample = list(metadata.values())[0]
print('article_path     :', sample.get('article_path',''))
print('full_article_path:', sample.get('full_article_path',''))

# Check all possible locations
rel = sample.get('full_article_path','')
candidates = [
    os.path.join(r'D:\Pics Can Lie', rel),
    os.path.join(r'D:\Pics Can Lie\kaggle_dataset_full', rel),
    os.path.join(r'D:\Pics Can Lie\kaggle_dataset_full', rel.replace('visual_news/', '')),
    os.path.join(r'D:\Pics Can Lie\kaggle_dataset_full\visual_news', rel.replace('visual_news/',''))
]

print('\nChecking paths:')
for c in candidates:
    print(f'  {"EXISTS" if os.path.exists(c) else "missing"} : {c}')

# Also check what's actually in kaggle_dataset_full
print('\nContents of kaggle_dataset_full:')
for item in os.listdir(r'D:\Pics Can Lie\kaggle_dataset_full'):
    print(f'  {item}')

article_path     : visual_news/origin/guardian/articles/931806.txt
full_article_path: visual_news/articles/guardian/http:::mobile-apps.guardianapis.com:items:film:2014:oct:06:will-poulter-interview-maze-runner-200.json

Checking paths:
  missing : D:\Pics Can Lie\visual_news/articles/guardian/http:::mobile-apps.guardianapis.com:items:film:2014:oct:06:will-poulter-interview-maze-runner-200.json
  missing : D:\Pics Can Lie\kaggle_dataset_full\visual_news/articles/guardian/http:::mobile-apps.guardianapis.com:items:film:2014:oct:06:will-poulter-interview-maze-runner-200.json
  missing : D:\Pics Can Lie\kaggle_dataset_full\articles/guardian/http:::mobile-apps.guardianapis.com:items:film:2014:oct:06:will-poulter-interview-maze-runner-200.json
  missing : D:\Pics Can Lie\kaggle_dataset_full\visual_news\articles/guardian/http:::mobile-apps.guardianapis.com:items:film:2014:oct:06:will-poulter-interview-maze-runner-200.json

Contents of kaggle_dataset_full:
  images
  merged_balanced
  metadata


In [1]:
import os

CLIP_FEATURES = r'D:\Pics Can Lie\clip_finetuned_v2\val_features'

print('Files in val_features folder:')
for f in os.listdir(CLIP_FEATURES):
    path = os.path.join(CLIP_FEATURES, f)
    size_mb = os.path.getsize(path) / 1024**2
    print(f'  {f:<40} {size_mb:>8.1f} MB')

# Also check the parent folder
print()
print('Files in clip_finetuned_v2 folder:')
parent = r'D:\Pics Can Lie\clip_finetuned_v2'
for f in os.listdir(parent):
    path = os.path.join(parent, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f'  {f:<40} {size_mb:>8.1f} MB')
    else:
        print(f'  {f}/ (folder)')

Files in val_features folder:
  clip_finetuned_probs.npy                      0.0 MB
  clip_finetuned_sims.npy                       0.0 MB
  clip_img_features.pt                         14.6 MB
  clip_txt_features.pt                         14.6 MB
  val_sample_ids.csv                            0.1 MB

Files in clip_finetuned_v2 folder:
  val_features/ (folder)


In [ ]:
import torch

img_feats = torch.load(r'D:\Pics Can Lie\clip_finetuned_v2\val_features\clip_img_features.pt')
txt_feats = torch.load(r'D:\Pics Can Lie\clip_finetuned_v2\val_features\clip_txt_features.pt')

print(f'Image features: {img_feats.shape} | dtype: {img_feats.dtype}')
print(f'Text features : {txt_feats.shape} | dtype: {txt_feats.dtype}')
print(f'Image norms (should be ~1.0): mean={img_feats.norm(dim=-1).mean():.4f}')
print(f'Text norms  (should be ~1.0): mean={txt_feats.norm(dim=-1).mean():.4f}')

In [3]:
import torch

img_feats = torch.load(r'D:\Pics Can Lie\clip_finetuned_v2\val_features\clip_img_features.pt')
txt_feats = torch.load(r'D:\Pics Can Lie\clip_finetuned_v2\val_features\clip_txt_features.pt')

print(f'Image features: {img_feats.shape} | dtype: {img_feats.dtype}')
print(f'Text features : {txt_feats.shape} | dtype: {txt_feats.dtype}')
print(f'Image norms (should be ~1.0): mean={img_feats.norm(dim=-1).mean():.4f}')
print(f'Text norms  (should be ~1.0): mean={txt_feats.norm(dim=-1).mean():.4f}')

Image features: torch.Size([5000, 768]) | dtype: torch.float32
Text features : torch.Size([5000, 768]) | dtype: torch.float32
Image norms (should be ~1.0): mean=15.1309
Text norms  (should be ~1.0): mean=12.1537


In [10]:
import json

with open(r"D:\Pics Can Lie\links_val.json", "r") as f:
    data = json.load(f)

print(f"Total samples: {len(data)}")

first_key = list(data.keys())[0]
sample = data[first_key]
print(f"\nSample keys: {list(sample.keys())}")
print(f"Label: {sample['label']}")
print(f"Direct search links: {len(sample.get('links_direct_search', []))}")
print(f"Inverse search links: {len(sample.get('links_inv_search', []))}")
print(f"Entities: {sample.get('entities', [])[:3]}")

has_direct  = sum(1 for s in data.values() if s.get('links_direct_search'))
has_inverse = sum(1 for s in data.values() if s.get('links_inv_search'))
print(f"\nSamples with direct search links: {has_direct}/{len(data)}")
print(f"Samples with inverse search links: {has_inverse}/{len(data)}")

avg_direct  = sum(len(s.get('links_direct_search', [])) for s in data.values()) / len(data)
avg_inverse = sum(len(s.get('links_inv_search', [])) for s in data.values()) / len(data)
print(f"Avg direct links per sample:  {avg_direct:.1f}")
print(f"Avg inverse links per sample: {avg_inverse:.1f}")

Total samples: 6972

Sample keys: ['label', 'entities', 'links_direct_search', 'links_inv_search']
Label: 0
Direct search links: 10
Inverse search links: 17
Entities: ['Agunnaryd', 'IKEA', 'IKEA']

Samples with direct search links: 6972/6972
Samples with inverse search links: 5079/6972
Avg direct links per sample:  9.3
Avg inverse links per sample: 5.5


In [11]:
import requests
import random
from tqdm import tqdm

# Sample 100 random direct links to test liveness
all_direct_links = []
for sample in data.values():
    for link_pair in sample.get('links_direct_search', []):
        all_direct_links.append(link_pair[0])  # image link is first item

test_links = random.sample(all_direct_links, min(100, len(all_direct_links)))

alive, dead, timeout = 0, 0, 0
for url in tqdm(test_links, desc="Testing links"):
    try:
        resp = requests.head(url, timeout=5, 
                            headers={"User-Agent": "Mozilla/5.0"})
        if resp.status_code < 400:
            alive += 1
        else:
            dead += 1
    except requests.exceptions.Timeout:
        timeout += 1
    except:
        dead += 1

total = alive + dead + timeout
print(f"\nLink liveness test (n=100):")
print(f"  Alive:   {alive}/{total} ({alive}%)")
print(f"  Dead:    {dead}/{total} ({dead}%)")
print(f"  Timeout: {timeout}/{total} ({timeout}%)")
print()
if alive >= 50:
    print("GOOD — enough links alive to proceed with crawling")
elif alive >= 30:
    print("MARGINAL — crawling possible but many gaps expected")
else:
    print("POOR — too many dead links, crawling not worth it")

Testing links: 100%|██████████| 100/100 [01:16<00:00,  1.31it/s]


Link liveness test (n=100):
  Alive:   70/100 (70%)
  Dead:    27/100 (27%)
  Timeout: 3/100 (3%)

GOOD — enough links alive to proceed with crawling


In [12]:
import pandas as pd

id_df = pd.read_csv(r"D:\Pics Can Lie\val_sample_ids.csv")
id_df["id"] = id_df["id"].astype(str)

first_keys = list(data.keys())[:5]
print(f"Links file key format: {first_keys}")
print(f"Your ID format: {id_df['id'].values[:5]}")

links_keys = set(data.keys())
your_ids   = set(id_df["id"].values)
overlap    = links_keys & your_ids
print(f"\nYour val IDs:    {len(your_ids)}")
print(f"Links file keys: {len(links_keys)}")
print(f"Overlap:         {len(overlap)}")
print(f"Coverage:        {len(overlap)/len(your_ids)*100:.1f}%")

Links file key format: ['0', '1', '2', '3', '4']
Your ID format: ['216004' '1085856' '1169411' '1117864' '324716']

Your val IDs:    3232
Links file keys: 6972
Overlap:         0
Coverage:        0.0%


In [13]:
import json, pandas as pd

# Load val annotations
VAL_ANN = r"D:\Pics Can Lie\dataset\data\NewsClipPings\merged_balanced\val.json"
with open(VAL_ANN, "r") as f:
    ann = json.load(f)

annotations = ann["annotations"]
print(f"Total val annotations: {len(annotations)}")
print(f"First annotation: {annotations[0]}")
print(f"Keys: {list(annotations[0].keys())}")

# Build index -> article_id mapping
# The links file index = position in annotations list
idx_to_id = {str(i): str(ann["id"]) for i, ann in enumerate(annotations)}

print(f"\nFirst 5 index->id mappings:")
for i in range(5):
    print(f"  {i} -> {idx_to_id[str(i)]}")

# Now check overlap with your val IDs
your_ids   = set(id_df["id"].values)
mapped_ids = set(idx_to_id.values())
overlap    = your_ids & mapped_ids

print(f"\nYour val IDs:        {len(your_ids)}")
print(f"Mapped annotation IDs: {len(mapped_ids)}")
print(f"Overlap:             {len(overlap)}")
print(f"Coverage:            {len(overlap)/len(your_ids)*100:.1f}%")

# Build the reverse mapping: article_id -> links data
id_to_links = {}
for idx, article_id in idx_to_id.items():
    if idx in data:
        id_to_links[article_id] = data[idx]

print(f"\nSamples with links mapped by article ID: {len(id_to_links)}")
your_covered = sum(1 for i in your_ids if i in id_to_links)
print(f"Your val samples covered: {your_covered}/{len(your_ids)} ({your_covered/len(your_ids)*100:.1f}%)")

Total val annotations: 7024
First annotation: {'id': 92318, 'image_id': 92318, 'similarity_score': 1, 'source_dataset': 2, 'falsified': False}
Keys: ['id', 'image_id', 'similarity_score', 'source_dataset', 'falsified']

First 5 index->id mappings:
  0 -> 92318
  1 -> 92318
  2 -> 224103
  3 -> 224103
  4 -> 367276

Your val IDs:        3232
Mapped annotation IDs: 3512
Overlap:             3232
Coverage:            100.0%

Samples with links mapped by article ID: 3487
Your val samples covered: 3209/3232 (99.3%)


In [14]:
print(f"Val samples in ID file: {len(id_df)}")
print(f"Path used: D:\\Pics Can Lie\\val_sample_ids.csv")

# Check if the 4090 version has 5000
import os
path_4090 = r"K:\Joee El Ghandour\Pics Can Lie\clip_finetuned_v2\val_features\val_sample_ids.csv"
if os.path.exists(path_4090):
    df2 = pd.read_csv(path_4090)
    print(f"Val samples in 4090 file: {len(df2)}")

Val samples in ID file: 5000
Path used: D:\Pics Can Lie\val_sample_ids.csv


In [15]:
import json, pandas as pd

# Load val annotations
VAL_ANN = r"D:\Pics Can Lie\dataset\data\NewsClipPings\merged_balanced\val.json"
with open(VAL_ANN, "r") as f:
    ann = json.load(f)

annotations = ann["annotations"]

# Build index -> article_id mapping
idx_to_id = {str(i): str(a["id"]) for i, a in enumerate(annotations)}

# Load your full 5000 val IDs
id_df = pd.read_csv(r"D:\Pics Can Lie\val_sample_ids.csv")
id_df["id"] = id_df["id"].astype(str)
your_ids = set(id_df["id"].values)

# Build article_id -> links mapping
id_to_links = {}
for idx, article_id in idx_to_id.items():
    if idx in data:
        id_to_links[article_id] = data[idx]

# Check coverage
your_covered = sum(1 for i in your_ids if i in id_to_links)
print(f"Your val samples:         {len(your_ids)}")
print(f"Samples with links:       {your_covered}/{len(your_ids)} ({your_covered/len(your_ids)*100:.1f}%)")
print(f"Samples without links:    {len(your_ids) - your_covered}")

# Check evidence availability
has_direct  = sum(1 for i in your_ids if i in id_to_links and id_to_links[i].get('links_direct_search'))
has_inverse = sum(1 for i in your_ids if i in id_to_links and id_to_links[i].get('links_inv_search'))
print(f"\nWith direct search links:  {has_direct}/{len(your_ids)}")
print(f"With inverse search links: {has_inverse}/{len(your_ids)}")

Your val samples:         3232
Samples with links:       3209/3232 (99.3%)
Samples without links:    23

With direct search links:  3209/3232
With inverse search links: 2328/3232


In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ── Reload everything ──
PROJECT_ROOT  = r'D:\Pics Can Lie'
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')


clip_probs = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_probs.npy'))
clip_sims  = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_sims.npy'))
id_df      = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels     = id_df['label'].values

img_feats = torch.load(os.path.join(CLIP_FEATURES, 'clip_img_features.pt'))
txt_feats = torch.load(os.path.join(CLIP_FEATURES, 'clip_txt_features.pt'))
img_feats = F.normalize(img_feats, dim=-1)
txt_feats = F.normalize(txt_feats, dim=-1)

deb_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'deberta_val_scores_v2.csv'))
deb_df['id'] = deb_df['id'].astype(str)
deb_lookup   = dict(zip(deb_df['id'], deb_df['entailment_score']))
deb_scores   = np.array([deb_lookup.get(i, 0.33) for i in id_df['id']])

ev_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'evidence_clip_scores.csv'))
ev_df['id'] = ev_df['id'].astype(str)
ev_lookup   = {row['id']: row for _, row in ev_df.iterrows()}
s2 = np.array([ev_lookup.get(i, {}).get('s2', 0.0) for i in id_df['id']])
s3 = np.array([ev_lookup.get(i, {}).get('s3', 0.0) for i in id_df['id']])
s4 = np.array([ev_lookup.get(i, {}).get('s4', 0.0) for i in id_df['id']])
s5 = np.array([ev_lookup.get(i, {}).get('s5', 0.0) for i in id_df['id']])
s6 = np.array([ev_lookup.get(i, {}).get('s6', 0.0) for i in id_df['id']])

wiki_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'wiki_nli_scores.csv'))
wiki_df['id'] = wiki_df['id'].astype(str)
wiki_lookup   = {row['id']: row for _, row in wiki_df.iterrows()}
w1 = np.array([wiki_lookup.get(i, {}).get('wiki_score', 0.33) for i in id_df['id']])

scalar_feats = np.column_stack([clip_probs, clip_sims, deb_scores, s2, s3, s4, s5, s6, w1])
scalar_feats_scaled = StandardScaler().fit_transform(scalar_feats)
scalar_tensor = torch.tensor(scalar_feats_scaled, dtype=torch.float32)

print('All signals loaded.')

# ── Rebuild AITR model ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class AITR(nn.Module):
    def __init__(self, embed_dim=768, scalar_dim=9, num_heads=8, num_layers=2,
                 dropout=0.3, hidden_dim=256):
        super().__init__()
        self.scalar_proj  = nn.Sequential(
            nn.Linear(scalar_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )
        self.type_embedding = nn.Embedding(5, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * 2, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.classifier  = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, img_emb, txt_emb, scalar):
        B = img_emb.size(0)
        prod_emb   = img_emb * txt_emb
        diff_emb   = img_emb - txt_emb
        scalar_emb = self.scalar_proj(scalar)
        tokens = torch.stack([img_emb, txt_emb, prod_emb, diff_emb, scalar_emb], dim=1)
        type_ids = torch.arange(5, device=tokens.device).unsqueeze(0).expand(B, -1)
        tokens   = tokens + self.type_embedding(type_ids)
        cls      = self.cls_token.expand(B, -1, -1)
        tokens   = torch.cat([cls, tokens], dim=1)
        out      = self.transformer(tokens)
        return self.classifier(out[:, 0]).squeeze(-1)

model = AITR().to(device)
model.load_state_dict(
    torch.load(os.path.join(PROJECT_ROOT, 'fusion_aitr', 'aitr_weights.pt'),
               map_location=device)
)
model.eval()
print('AITR model loaded.')

# ── Get val split (same random seed as training) ──
indices = np.arange(len(labels))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=labels)

img_val    = img_feats[val_idx]
txt_val    = txt_feats[val_idx]
scl_val    = scalar_tensor[val_idx]
y_val      = labels[val_idx]

val_ds  = TensorDataset(img_val, txt_val, scl_val,
                         torch.tensor(y_val, dtype=torch.float32))
val_ld  = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

# ── Collect all probabilities ──
all_probs, all_labels = [], []
with torch.no_grad():
    for img, txt, scl, yb in val_ld:
        img, txt, scl = img.to(device), txt.to(device), scl.to(device)
        logits = model(img, txt, scl)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(yb.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels, dtype=int)

# ── Threshold sweep ──
print()
print('=' * 75)
print('THRESHOLD SWEEP')
print('=' * 75)
print(f'{"Threshold":>10} | {"Accuracy":>9} | {"F1":>7} | {"REAL recall":>12} | {"FAKE recall":>12}')
print('-' * 75)

best_acc, best_f1, best_thr_acc, best_thr_f1 = 0, 0, 0.5, 0.5
results = []

for thr in np.arange(0.25, 0.76, 0.01):
    preds     = (all_probs > thr).astype(int)
    acc       = accuracy_score(all_labels, preds)
    f1        = f1_score(all_labels, preds)
    real_rec  = (preds[all_labels==0] == 0).mean()
    fake_rec  = (preds[all_labels==1] == 1).mean()
    results.append((thr, acc, f1, real_rec, fake_rec))

    if acc > best_acc:
        best_acc, best_thr_acc = acc, thr
    if f1 > best_f1:
        best_f1, best_thr_f1 = f1, thr

# Print every 0.05 for readability
for thr, acc, f1, real_rec, fake_rec in results:
    if abs(thr % 0.05) < 0.005 or abs(thr - best_thr_acc) < 0.005:
        marker = ' <-- best acc' if abs(thr - best_thr_acc) < 0.005 else ''
        marker = ' <-- best F1' if abs(thr - best_thr_f1) < 0.005 else marker
        print(f'{thr:>10.2f} | {acc*100:>8.2f}% | {f1:>7.4f} | {real_rec:>12.4f} | {fake_rec:>12.4f}{marker}')

print()
print(f'Best Accuracy: {best_acc*100:.2f}% at threshold {best_thr_acc:.2f}')
print(f'Best F1      : {best_f1:.4f} at threshold {best_thr_f1:.2f}')
print(f'Default (0.5): {accuracy_score(all_labels, (all_probs>0.5).astype(int))*100:.2f}%')
print(f'Gain         : +{(best_acc - accuracy_score(all_labels, (all_probs>0.5).astype(int)))*100:.2f}%')

# ── Full report at best threshold ──
print()
print(f'Classification report at best accuracy threshold ({best_thr_acc:.2f}):')
print(classification_report(all_labels, (all_probs > best_thr_acc).astype(int),
                              target_names=['REAL', 'FAKE']))

All signals loaded.


d:\Pics Can Lie\venv\lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


AITR model loaded.

THRESHOLD SWEEP
 Threshold |  Accuracy |      F1 |  REAL recall |  FAKE recall
---------------------------------------------------------------------------
      0.30 |    85.90% |  0.8710 |       0.7660 |       0.9520
      0.35 |    86.60% |  0.8752 |       0.7920 |       0.9400
      0.40 |    86.60% |  0.8738 |       0.8040 |       0.9280
      0.45 |    87.20% |  0.8779 |       0.8240 |       0.9200
      0.50 |    87.50% |  0.8799 |       0.8340 |       0.9160
      0.54 |    88.30% |  0.8859 |       0.8580 |       0.9080 <-- best F1
      0.55 |    88.00% |  0.8826 |       0.8580 |       0.9020
      0.60 |    87.80% |  0.8775 |       0.8820 |       0.8740
      0.65 |    86.80% |  0.8639 |       0.8980 |       0.8380
      0.70 |    86.10% |  0.8526 |       0.9180 |       0.8040
      0.75 |    83.90% |  0.8217 |       0.9360 |       0.7420

Best Accuracy: 88.30% at threshold 0.54
Best F1      : 0.8859 at threshold 0.54
Default (0.5): 87.50%
Gain         : +0

In [2]:
import json
result = {
    'optimal_threshold': 0.54,
    'accuracy_at_threshold': 0.8830,
    'f1_at_threshold': 0.8859,
    'default_threshold_accuracy': 0.8750,
    'gain': 0.0080,
}
with open(r'D:\Pics Can Lie\fusion_aitr\optimal_threshold.json', 'w') as f:
    json.dump(result, f, indent=2)
print('Saved.')

Saved.


In [1]:
import os
import psutil
import torch

ram = psutil.virtual_memory()
print(f'RAM available: {ram.available / 1024**3:.1f} GB / {ram.total / 1024**3:.1f} GB')

if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM total: {total:.1f} GB')
else:
    print('No CUDA')

# Check file sizes
PROJECT_ROOT  = r'D:\Pics Can Lie'
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')

for f in os.listdir(CLIP_FEATURES):
    path = os.path.join(CLIP_FEATURES, f)
    print(f'  {f:<40} {os.path.getsize(path)/1024**2:.1f} MB')

RAM available: 2.1 GB / 15.7 GB
VRAM total: 8.0 GB
  clip_finetuned_probs.npy                 0.0 MB
  clip_finetuned_sims.npy                  0.0 MB
  clip_img_features.pt                     14.6 MB
  clip_txt_features.pt                     14.6 MB
  val_sample_ids.csv                       0.1 MB


In [30]:
import psutil
ram = psutil.virtual_memory()
print(f'RAM available: {ram.available / 1024**3:.1f} GB / {ram.total / 1024**3:.1f} GB')

RAM available: 6.2 GB / 15.7 GB


In [33]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import clip as clip_lib
import json, warnings
warnings.filterwarnings('ignore')

# ── Paths ──
PROJECT_ROOT  = r'D:\Pics Can Lie'
DATASET_ROOT  = os.path.join(PROJECT_ROOT, 'kaggle_dataset_full')
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')
CLIP_MODEL_DIR = os.path.join(PROJECT_ROOT, 'models', 'clip')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

# ── Load scalar signals ──
id_df       = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels      = id_df['label'].values

clip_probs  = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_probs.npy'))
clip_sims   = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_sims.npy'))

deb_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'deberta_val_scores_v2.csv'))
deb_df['id'] = deb_df['id'].astype(str)
deb_lookup   = dict(zip(deb_df['id'], deb_df['entailment_score']))
deb_scores   = np.array([deb_lookup.get(i, 0.33) for i in id_df['id']])

ev_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'evidence_clip_scores.csv'))
ev_df['id'] = ev_df['id'].astype(str)
ev_lookup   = {row['id']: row for _, row in ev_df.iterrows()}
s2 = np.array([ev_lookup.get(i, {}).get('s2', 0.0) for i in id_df['id']])
s3 = np.array([ev_lookup.get(i, {}).get('s3', 0.0) for i in id_df['id']])
s4 = np.array([ev_lookup.get(i, {}).get('s4', 0.0) for i in id_df['id']])
s5 = np.array([ev_lookup.get(i, {}).get('s5', 0.0) for i in id_df['id']])
s6 = np.array([ev_lookup.get(i, {}).get('s6', 0.0) for i in id_df['id']])

wiki_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'wiki_nli_scores.csv'))
wiki_df['id'] = wiki_df['id'].astype(str)
wiki_lookup   = {row['id']: row for _, row in wiki_df.iterrows()}
w1 = np.array([wiki_lookup.get(i, {}).get('wiki_score', 0.33) for i in id_df['id']])

scalar_feats        = np.column_stack([clip_probs, clip_sims, deb_scores, s2, s3, s4, s5, s6, w1])
scaler              = StandardScaler()
scalar_feats_scaled = scaler.fit_transform(scalar_feats)
scalar_tensor       = torch.tensor(scalar_feats_scaled, dtype=torch.float32).to(device)

# ── Load CLIP embeddings ──
img_feats = F.normalize(torch.load(os.path.join(CLIP_FEATURES, 'clip_img_features.pt')), dim=-1).to(device)
txt_feats = F.normalize(torch.load(os.path.join(CLIP_FEATURES, 'clip_txt_features.pt')), dim=-1).to(device)

print(f'Loaded: {len(id_df)} samples')
print(f'img_feats: {img_feats.shape} | txt_feats: {txt_feats.shape}')

# ── Load AITR model ──
class AITR(nn.Module):
    def __init__(self, embed_dim=768, scalar_dim=9, num_heads=8, num_layers=2,
                 dropout=0.3, hidden_dim=256):
        super().__init__()
        self.scalar_proj = nn.Sequential(
            nn.Linear(scalar_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )
        self.type_embedding = nn.Embedding(5, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * 2, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.classifier  = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, img_emb, txt_emb, scalar):
        B = img_emb.size(0)
        prod_emb   = img_emb * txt_emb
        diff_emb   = img_emb - txt_emb
        scalar_emb = self.scalar_proj(scalar)
        tokens     = torch.stack([img_emb, txt_emb, prod_emb, diff_emb, scalar_emb], dim=1)
        type_ids   = torch.arange(5, device=tokens.device).unsqueeze(0).expand(B, -1)
        tokens     = tokens + self.type_embedding(type_ids)
        cls        = self.cls_token.expand(B, -1, -1)
        tokens     = torch.cat([cls, tokens], dim=1)
        out        = self.transformer(tokens)
        return self.classifier(out[:, 0]).squeeze(-1)

aitr_model = AITR().to(device)
aitr_model.load_state_dict(
    torch.load(os.path.join(PROJECT_ROOT, 'fusion_aitr', 'aitr_weights.pt'),
               map_location=device)
)
aitr_model.eval()
print('AITR loaded.')

# ── Load CLIP for TTA ──
os.environ['CLIP_DOWNLOAD_ROOT'] = CLIP_MODEL_DIR
print('Loading CLIP ViT-L/14...')
clip_model, _ = clip_lib.load('ViT-L/14', device=device, jit=False)
clip_model = clip_model.float().eval()
for p in clip_model.parameters():
    p.requires_grad = False
print('CLIP loaded.')

# ── TTA transforms ──
_CLIP_MEAN = [0.48145466, 0.4578275,  0.40821073]
_CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]

tta_transforms = [
    transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(_CLIP_MEAN, _CLIP_STD),
    ]),
    transforms.Compose([
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize(_CLIP_MEAN, _CLIP_STD),
    ]),
    transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(_CLIP_MEAN, _CLIP_STD),
    ]),
    transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(_CLIP_MEAN, _CLIP_STD),
    ]),
    transforms.Compose([
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(_CLIP_MEAN, _CLIP_STD),
    ]),
]

# ── Load val metadata ──
with open(os.path.join(DATASET_ROOT, 'metadata', 'val.json'), 'r') as f:
    metadata = json.load(f)

def resolve_image_path(rel_path):
    return os.path.join(DATASET_ROOT, str(rel_path).replace('visual_news/', 'images/'))

# ── TTA loop ──
print(f'\nRunning TTA ({len(tta_transforms)} augmentations × {len(id_df)} samples)...')

tta_img_feats = torch.zeros(len(id_df), 768, device=device)
failed = 0

for i, sample_id in enumerate(tqdm(id_df['id'].values, desc='TTA')):
    meta     = metadata.get(sample_id, {})
    img_path = resolve_image_path(meta.get('image_path', ''))

    aug_feats = []
    for transform in tta_transforms:
        try:
            img = Image.open(img_path).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = clip_model.encode_image(img_tensor).float()
                feat = F.normalize(feat, dim=-1)
            aug_feats.append(feat.squeeze(0))
        except:
            aug_feats.append(img_feats[i])
            failed += 1
            break

    stacked = torch.stack(aug_feats).mean(dim=0)
    tta_img_feats[i] = F.normalize(stacked.unsqueeze(0), dim=-1).squeeze(0)

print(f'TTA complete. Failed: {failed}')
torch.save(tta_img_feats.cpu(), os.path.join(PROJECT_ROOT, 'tta_img_features.pt'))
print('TTA features saved.')

# ── AITR inference with TTA features ──
print('\nRunning AITR with TTA features...')

indices = np.arange(len(labels))
_, val_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels)

img_val_tta = tta_img_feats[val_idx]
txt_val     = txt_feats[val_idx]
scl_val     = scalar_tensor[val_idx]
y_val       = labels[val_idx]

from torch.utils.data import TensorDataset
val_ds = TensorDataset(img_val_tta, txt_val, scl_val,
                        torch.tensor(y_val, dtype=torch.float32))
val_ld = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

all_probs, all_preds, all_labels = [], [], []
aitr_model.eval()
with torch.no_grad():
    for img, txt, scl, yb in val_ld:
        img, txt, scl = img.to(device), txt.to(device), scl.to(device)
        logits = aitr_model(img, txt, scl)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend((probs > 0.5).astype(int))
        all_labels.extend(yb.long().numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

# ── Threshold sweep on TTA probs ──
print()
print('=' * 65)
print('TTA RESULTS — THRESHOLD SWEEP')
print('=' * 65)
print(f'{"Threshold":>10} | {"Accuracy":>9} | {"F1":>7} | {"REAL rec":>9} | {"FAKE rec":>9}')
print('-' * 65)

best_acc, best_thr = 0, 0.5
for thr in np.arange(0.30, 0.76, 0.05):
    preds    = (all_probs > thr).astype(int)
    acc      = accuracy_score(all_labels, preds)
    f1       = f1_score(all_labels, preds)
    real_rec = (preds[all_labels==0] == 0).mean()
    fake_rec = (preds[all_labels==1] == 1).mean()
    marker   = ' <-- best' if acc > best_acc else ''
    if acc > best_acc:
        best_acc, best_thr = acc, thr
    print(f'{thr:>10.2f} | {acc*100:>8.2f}% | {f1:>7.4f} | {real_rec:>9.4f} | {fake_rec:>9.4f}{marker}')

print()
print(f'Best TTA accuracy : {best_acc*100:.2f}% at threshold {best_thr:.2f}')
print(f'Previous best     : 88.30% (AITR no TTA, threshold 0.54)')
print(f'Gain from TTA     : {(best_acc - 0.8830)*100:+.2f}%')
print()
print(f'Classification report at threshold {best_thr:.2f}:')
print(classification_report(all_labels, (all_probs > best_thr).astype(int),
                              target_names=['REAL', 'FAKE']))

Device: cuda
GPU  : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM : 8.0 GB
Loaded: 5000 samples
img_feats: torch.Size([5000, 768]) | txt_feats: torch.Size([5000, 768])
AITR loaded.
Loading CLIP ViT-L/14...
CLIP loaded.

Running TTA (5 augmentations × 5000 samples)...


TTA: 100%|██████████| 5000/5000 [35:47<00:00,  2.33it/s]


TTA complete. Failed: 0
TTA features saved.

Running AITR with TTA features...

TTA RESULTS — THRESHOLD SWEEP
 Threshold |  Accuracy |      F1 |  REAL rec |  FAKE rec
-----------------------------------------------------------------
      0.30 |    85.80% |  0.8700 |    0.7660 |    0.9500 <-- best
      0.35 |    86.70% |  0.8760 |    0.7940 |    0.9400 <-- best
      0.40 |    86.70% |  0.8749 |    0.8040 |    0.9300
      0.45 |    87.20% |  0.8779 |    0.8240 |    0.9200 <-- best
      0.50 |    87.50% |  0.8799 |    0.8340 |    0.9160 <-- best
      0.55 |    88.00% |  0.8826 |    0.8580 |    0.9020 <-- best
      0.60 |    87.80% |  0.8775 |    0.8820 |    0.8740
      0.65 |    86.80% |  0.8639 |    0.8980 |    0.8380
      0.70 |    86.10% |  0.8526 |    0.9180 |    0.8040
      0.75 |    83.90% |  0.8217 |    0.9360 |    0.7420

Best TTA accuracy : 88.00% at threshold 0.55
Previous best     : 88.30% (AITR no TTA, threshold 0.54)
Gain from TTA     : -0.30%

Classification report

In [34]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

# ── Reload signals ──
PROJECT_ROOT  = r'D:\Pics Can Lie'
CLIP_FEATURES = r'D:\Pics Can Lie\clip_finetuned_v2\val_features'

clip_probs = np.load(f'{CLIP_FEATURES}/clip_finetuned_probs.npy')
clip_sims  = np.load(f'{CLIP_FEATURES}/clip_finetuned_sims.npy')
id_df      = pd.read_csv(f'{CLIP_FEATURES}/val_sample_ids.csv')
id_df['id'] = id_df['id'].astype(str)
labels     = id_df['label'].values

deb_df     = pd.read_csv(f'{PROJECT_ROOT}/deberta_val_scores_v2.csv')
deb_df['id'] = deb_df['id'].astype(str)
deb_lookup = dict(zip(deb_df['id'], deb_df['entailment_score']))
deb_scores = np.array([deb_lookup.get(i, 0.33) for i in id_df['id']])

ev_df      = pd.read_csv(f'{PROJECT_ROOT}/evidence_clip_scores.csv')
ev_df['id'] = ev_df['id'].astype(str)
ev_lookup  = {row['id']: row for _, row in ev_df.iterrows()}
s2 = np.array([ev_lookup.get(i, {}).get('s2', 0.0) for i in id_df['id']])
s3 = np.array([ev_lookup.get(i, {}).get('s3', 0.0) for i in id_df['id']])
s4 = np.array([ev_lookup.get(i, {}).get('s4', 0.0) for i in id_df['id']])
s5 = np.array([ev_lookup.get(i, {}).get('s5', 0.0) for i in id_df['id']])
s6 = np.array([ev_lookup.get(i, {}).get('s6', 0.0) for i in id_df['id']])

wiki_df    = pd.read_csv(f'{PROJECT_ROOT}/wiki_nli_scores.csv')
wiki_df['id'] = wiki_df['id'].astype(str)
wiki_lookup = {row['id']: row for _, row in wiki_df.iterrows()}
w1 = np.array([wiki_lookup.get(i, {}).get('wiki_score', 0.33) for i in id_df['id']])

X = np.column_stack([clip_probs, clip_sims, deb_scores, s2, s3, s4, s5, s6, w1])

# ── Train XGBoost ──
X_train, X_val, y_train, y_val = train_test_split(
    X, labels, test_size=0.2, random_state=42, stratify=labels)

xgb = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8,
                     eval_metric='logloss', early_stopping_rounds=30,
                     random_state=42)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

xgb_probs = xgb.predict_proba(X_val)[:, 1]

# ── XGBoost threshold sweep ──
print('=' * 60)
print('XGBoost threshold sweep')
print('=' * 60)
best_acc, best_thr = 0, 0.5
for thr in np.arange(0.30, 0.76, 0.05):
    preds = (xgb_probs > thr).astype(int)
    acc   = accuracy_score(y_val, preds)
    f1    = f1_score(y_val, preds)
    if acc > best_acc:
        best_acc, best_thr = acc, thr
    print(f'  thr={thr:.2f}  acc={acc*100:.2f}%  f1={f1:.4f}')

print(f'\nBest XGBoost: {best_acc*100:.2f}% at threshold {best_thr:.2f}')

# ── Load AITR probs (same val split) ──
# Re-run AITR inference to get probs on same val_idx
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

img_feats = F.normalize(torch.load(f'{CLIP_FEATURES}/clip_img_features.pt'), dim=-1)
txt_feats = F.normalize(torch.load(f'{CLIP_FEATURES}/clip_txt_features.pt'), dim=-1)

scaler = StandardScaler()
scalar_feats_scaled = scaler.fit_transform(X)
scalar_tensor = torch.tensor(scalar_feats_scaled, dtype=torch.float32)

indices = np.arange(len(labels))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels)

class AITR(nn.Module):
    def __init__(self, embed_dim=768, scalar_dim=9, num_heads=8, num_layers=2,
                 dropout=0.3, hidden_dim=256):
        super().__init__()
        self.scalar_proj = nn.Sequential(nn.Linear(scalar_dim, embed_dim), nn.LayerNorm(embed_dim))
        self.type_embedding = nn.Embedding(5, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*2,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.classifier  = nn.Sequential(
            nn.LayerNorm(embed_dim), nn.Linear(embed_dim, hidden_dim),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
    def forward(self, img_emb, txt_emb, scalar):
        B = img_emb.size(0)
        tokens = torch.stack([img_emb, txt_emb, img_emb*txt_emb, img_emb-txt_emb,
                               self.scalar_proj(scalar)], dim=1)
        type_ids = torch.arange(5, device=tokens.device).unsqueeze(0).expand(B, -1)
        tokens   = tokens + self.type_embedding(type_ids)
        cls      = self.cls_token.expand(B, -1, -1)
        out      = self.transformer(torch.cat([cls, tokens], dim=1))
        return self.classifier(out[:, 0]).squeeze(-1)

aitr_model = AITR().to(device)
aitr_model.load_state_dict(torch.load(
    f'{PROJECT_ROOT}/fusion_aitr/aitr_weights.pt', map_location=device))
aitr_model.eval()

val_ds = TensorDataset(img_feats[val_idx].to(device), txt_feats[val_idx].to(device),
                        scalar_tensor[val_idx].to(device),
                        torch.tensor(y_val, dtype=torch.float32))
val_ld = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

aitr_probs = []
with torch.no_grad():
    for img, txt, scl, yb in val_ld:
        logits = aitr_model(img, txt, scl)
        aitr_probs.extend(torch.sigmoid(logits).cpu().numpy())
aitr_probs = np.array(aitr_probs)

# ── Ensemble XGBoost + AITR ──
print()
print('=' * 60)
print('ENSEMBLE XGBoost + AITR — threshold sweep')
print('=' * 60)

best_ens_acc, best_ens_thr, best_w = 0, 0.5, 0.5
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    ens_probs = w * aitr_probs + (1 - w) * xgb_probs
    for thr in np.arange(0.40, 0.70, 0.02):
        preds = (ens_probs > thr).astype(int)
        acc   = accuracy_score(y_val, preds)
        if acc > best_ens_acc:
            best_ens_acc, best_ens_thr, best_w = acc, thr, w

print(f'Best ensemble: {best_ens_acc*100:.2f}% '
      f'(AITR weight={best_w:.1f}, XGB weight={1-best_w:.1f}, threshold={best_ens_thr:.2f})')
print()
print('=' * 60)
print('FINAL COMPARISON')
print('=' * 60)
print(f'  XGBoost alone (tuned threshold)  : {best_acc*100:.2f}%')
print(f'  AITR alone (threshold 0.54)      : 88.30%')
print(f'  Ensemble XGBoost + AITR          : {best_ens_acc*100:.2f}%')
print()
print(f'Classification report (best ensemble):')
best_ens_probs = best_w * aitr_probs + (1 - best_w) * xgb_probs
print(classification_report(y_val, (best_ens_probs > best_ens_thr).astype(int),
                              target_names=['REAL', 'FAKE']))

XGBoost threshold sweep
  thr=0.30  acc=86.30%  f1=0.8730
  thr=0.35  acc=86.70%  f1=0.8746
  thr=0.40  acc=87.20%  f1=0.8769
  thr=0.45  acc=87.30%  f1=0.8766
  thr=0.50  acc=87.50%  f1=0.8764
  thr=0.55  acc=86.90%  f1=0.8678
  thr=0.60  acc=86.50%  f1=0.8621
  thr=0.65  acc=86.20%  f1=0.8556
  thr=0.70  acc=85.30%  f1=0.8404
  thr=0.75  acc=84.40%  f1=0.8259

Best XGBoost: 87.50% at threshold 0.50

ENSEMBLE XGBoost + AITR — threshold sweep
Best ensemble: 87.90% (AITR weight=0.6, XGB weight=0.4, threshold=0.52)

FINAL COMPARISON
  XGBoost alone (tuned threshold)  : 87.50%
  AITR alone (threshold 0.54)      : 88.30%
  Ensemble XGBoost + AITR          : 87.90%

Classification report (best ensemble):
              precision    recall  f1-score   support

        REAL       0.89      0.86      0.88       500
        FAKE       0.87      0.90      0.88       500

    accuracy                           0.88      1000
   macro avg       0.88      0.88      0.88      1000
weighted avg       

In [35]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression

# Calibrate AITR probs using val split
# Use first 80% as calibration train, last 20% as test
cal_train_idx = int(len(aitr_probs) * 0.7)

cal_X_train = aitr_probs[:cal_train_idx].reshape(-1, 1)
cal_y_train = y_val[:cal_train_idx]
cal_X_test  = aitr_probs[cal_train_idx:].reshape(-1, 1)
cal_y_test  = y_val[cal_train_idx:]

# Platt scaling
platt = LogisticRegression()
platt.fit(cal_X_train, cal_y_train)
cal_probs = platt.predict_proba(cal_X_test)[:, 1]

# Threshold sweep on calibrated probs
print('=' * 55)
print('Calibrated AITR threshold sweep')
print('=' * 55)
best_cal_acc, best_cal_thr = 0, 0.5
for thr in np.arange(0.30, 0.76, 0.02):
    preds = (cal_probs > thr).astype(int)
    acc   = accuracy_score(cal_y_test, preds)
    f1    = f1_score(cal_y_test, preds)
    if acc > best_cal_acc:
        best_cal_acc, best_cal_thr = acc, thr
    if abs(thr % 0.05) < 0.01:
        print(f'  thr={thr:.2f}  acc={acc*100:.2f}%  f1={f1:.4f}')

print(f'\nBest calibrated: {best_cal_acc*100:.2f}% at threshold {best_cal_thr:.2f}')
print(f'Previous best  : 88.30%')
print(f'Gain           : {(best_cal_acc - 0.8830)*100:+.2f}%')

Calibrated AITR threshold sweep
  thr=0.40  acc=86.00%  f1=0.8671
  thr=0.50  acc=87.33%  f1=0.8774
  thr=0.60  acc=87.00%  f1=0.8687
  thr=0.70  acc=86.00%  f1=0.8521

Best calibrated: 88.00% at threshold 0.56
Previous best  : 88.30%
Gain           : -0.30%


In [36]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import json, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT  = r'D:\Pics Can Lie'
DATASET_ROOT  = os.path.join(PROJECT_ROOT, 'kaggle_dataset_full')
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load signals ──
id_df       = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels      = id_df['label'].values

clip_probs  = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_probs.npy'))
clip_sims   = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_sims.npy'))

deb_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'deberta_val_scores_v2.csv'))
deb_df['id'] = deb_df['id'].astype(str)
deb_lookup   = dict(zip(deb_df['id'], deb_df['entailment_score']))
deb_scores   = np.array([deb_lookup.get(i, 0.33) for i in id_df['id']])

ev_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'evidence_clip_scores.csv'))
ev_df['id'] = ev_df['id'].astype(str)
ev_lookup   = {row['id']: row for _, row in ev_df.iterrows()}
s2 = np.array([ev_lookup.get(i, {}).get('s2', 0.0) for i in id_df['id']])
s3 = np.array([ev_lookup.get(i, {}).get('s3', 0.0) for i in id_df['id']])
s4 = np.array([ev_lookup.get(i, {}).get('s4', 0.0) for i in id_df['id']])
s5 = np.array([ev_lookup.get(i, {}).get('s5', 0.0) for i in id_df['id']])
s6 = np.array([ev_lookup.get(i, {}).get('s6', 0.0) for i in id_df['id']])

wiki_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'wiki_nli_scores.csv'))
wiki_df['id'] = wiki_df['id'].astype(str)
wiki_lookup   = {row['id']: row for _, row in wiki_df.iterrows()}
w1           = np.array([wiki_lookup.get(i, {}).get('wiki_score',   0.33) for i in id_df['id']])
entity_count = np.array([wiki_lookup.get(i, {}).get('entity_count', 0)    for i in id_df['id']])
wiki_cov     = np.array([wiki_lookup.get(i, {}).get('wiki_coverage', 0)   for i in id_df['id']])

X = np.column_stack([clip_probs, clip_sims, deb_scores, s2, s3, s4, s5, s6, w1])

# ── Load metadata ──
with open(os.path.join(DATASET_ROOT, 'metadata', 'val.json'), 'r') as f:
    metadata = json.load(f)

# ── Load AITR and get val probs ──
class AITR(nn.Module):
    def __init__(self, embed_dim=768, scalar_dim=9, num_heads=8, num_layers=2,
                 dropout=0.3, hidden_dim=256):
        super().__init__()
        self.scalar_proj    = nn.Sequential(nn.Linear(scalar_dim, embed_dim), nn.LayerNorm(embed_dim))
        self.type_embedding = nn.Embedding(5, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*2,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.classifier  = nn.Sequential(
            nn.LayerNorm(embed_dim), nn.Linear(embed_dim, hidden_dim),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
    def forward(self, img_emb, txt_emb, scalar):
        B = img_emb.size(0)
        tokens   = torch.stack([img_emb, txt_emb, img_emb*txt_emb,
                                  img_emb-txt_emb, self.scalar_proj(scalar)], dim=1)
        type_ids = torch.arange(5, device=tokens.device).unsqueeze(0).expand(B, -1)
        tokens   = tokens + self.type_embedding(type_ids)
        cls      = self.cls_token.expand(B, -1, -1)
        out      = self.transformer(torch.cat([cls, tokens], dim=1))
        return self.classifier(out[:, 0]).squeeze(-1)

img_feats     = F.normalize(torch.load(os.path.join(CLIP_FEATURES, 'clip_img_features.pt')), dim=-1)
txt_feats     = F.normalize(torch.load(os.path.join(CLIP_FEATURES, 'clip_txt_features.pt')), dim=-1)
scaler        = StandardScaler()
scalar_scaled = torch.tensor(scaler.fit_transform(X), dtype=torch.float32)

aitr_model = AITR().to(device)
aitr_model.load_state_dict(torch.load(
    os.path.join(PROJECT_ROOT, 'fusion_aitr', 'aitr_weights.pt'), map_location=device))
aitr_model.eval()

indices = np.arange(len(labels))
_, val_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels)

val_ds = TensorDataset(img_feats[val_idx].to(device), txt_feats[val_idx].to(device),
                        scalar_scaled[val_idx].to(device),
                        torch.tensor(labels[val_idx], dtype=torch.float32))
val_ld = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

all_probs, all_labels_val = [], []
with torch.no_grad():
    for img, txt, scl, yb in val_ld:
        logits = aitr_model(img, txt, scl)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels_val.extend(yb.long().numpy())

all_probs      = np.array(all_probs)
all_labels_val = np.array(all_labels_val)
THRESHOLD      = 0.54
all_preds      = (all_probs > THRESHOLD).astype(int)

# ── Build per-sample error dataframe ──
val_ids = id_df['id'].values[val_idx]

rows = []
for i, (sid, prob, pred, true) in enumerate(zip(val_ids, all_probs, all_preds, all_labels_val)):
    meta    = metadata.get(sid, {})
    correct = int(pred == true)
    error_type = None
    if not correct:
        error_type = 'FP' if pred == 1 else 'FN'  # FP=real→fake, FN=fake→real

    rows.append({
        'id'          : sid,
        'true_label'  : true,           # 0=real, 1=fake
        'pred_label'  : pred,
        'prob'        : prob,
        'correct'     : correct,
        'error_type'  : error_type,
        'confidence'  : abs(prob - 0.5) * 2,  # 0=uncertain, 1=max confident
        'source'      : meta.get('source', 'unknown'),
        'topic'       : meta.get('topic', 'unknown'),
        'has_person'  : meta.get('image_has_person', False),
        'entity_count': entity_count[val_idx[i]],
        'wiki_cov'    : wiki_cov[val_idx[i]],
        'clip_prob'   : clip_probs[val_idx[i]],
        'clip_sim'    : clip_sims[val_idx[i]],
        's2'          : s2[val_idx[i]],
        'caption'     : meta.get('caption', '')[:100],
    })

err_df = pd.DataFrame(rows)
errors = err_df[err_df['correct'] == 0]
correct_df = err_df[err_df['correct'] == 1]

print(f'Val samples  : {len(err_df)}')
print(f'Correct      : {len(correct_df)} ({len(correct_df)/len(err_df)*100:.1f}%)')
print(f'Errors       : {len(errors)} ({len(errors)/len(err_df)*100:.1f}%)')
print(f'  FP (real→fake): {(errors["error_type"]=="FP").sum()}')
print(f'  FN (fake→real): {(errors["error_type"]=="FN").sum()}')

# ── Analysis 1: Error rate by news source ──
print()
print('=' * 60)
print('ERROR RATE BY NEWS SOURCE')
print('=' * 60)
source_stats = err_df.groupby('source').agg(
    total=('correct', 'count'),
    errors=('correct', lambda x: (x==0).sum()),
    accuracy=('correct', 'mean')
).sort_values('accuracy')
source_stats['error_rate'] = 1 - source_stats['accuracy']
print(source_stats[source_stats['total'] >= 10].to_string())

# ── Analysis 2: Error rate by topic ──
print()
print('=' * 60)
print('ERROR RATE BY TOPIC')
print('=' * 60)
topic_stats = err_df.groupby('topic').agg(
    total=('correct', 'count'),
    errors=('correct', lambda x: (x==0).sum()),
    accuracy=('correct', 'mean')
).sort_values('accuracy')
print(topic_stats[topic_stats['total'] >= 5].to_string())

# ── Analysis 3: Images with vs without people ──
print()
print('=' * 60)
print('ERROR RATE: IMAGES WITH vs WITHOUT PEOPLE')
print('=' * 60)
for has_person in [True, False]:
    subset = err_df[err_df['has_person'] == has_person]
    acc    = subset['correct'].mean()
    print(f'  has_person={has_person}: {len(subset)} samples | acc={acc*100:.2f}% | errors={int((1-acc)*len(subset))}')

# ── Analysis 4: Confidence of errors ──
print()
print('=' * 60)
print('ERROR CONFIDENCE ANALYSIS')
print('=' * 60)
print(f'  Mean confidence of CORRECT predictions : {correct_df["confidence"].mean():.3f}')
print(f'  Mean confidence of WRONG predictions   : {errors["confidence"].mean():.3f}')
print()
# High-confidence errors (the worst kind)
high_conf_errors = errors[errors['confidence'] > 0.5]
print(f'  High-confidence errors (conf > 0.5): {len(high_conf_errors)}')
print(f'    FP (real→fake): {(high_conf_errors["error_type"]=="FP").sum()}')
print(f'    FN (fake→real): {(high_conf_errors["error_type"]=="FN").sum()}')

# ── Analysis 5: Entity count effect ──
print()
print('=' * 60)
print('ENTITY COUNT EFFECT ON ACCURACY')
print('=' * 60)
for n in [0, 1, 2, 3]:
    subset = err_df[err_df['entity_count'] == n]
    if len(subset) > 5:
        acc = subset['correct'].mean()
        print(f'  entity_count={n}: {len(subset):4d} samples | acc={acc*100:.2f}%')
subset = err_df[err_df['entity_count'] >= 4]
if len(subset) > 5:
    acc = subset['correct'].mean()
    print(f'  entity_count>=4: {len(subset):3d} samples | acc={acc*100:.2f}%')

# ── Analysis 6: Hardest samples (most confident wrong predictions) ──
print()
print('=' * 60)
print('TOP 10 HARDEST ERRORS (most confident wrong predictions)')
print('=' * 60)
hardest = errors.nlargest(10, 'confidence')[
    ['id', 'true_label', 'pred_label', 'prob', 'confidence', 'source', 'topic', 'caption']
]
for _, row in hardest.iterrows():
    true_str = 'REAL' if row['true_label'] == 0 else 'FAKE'
    pred_str = 'REAL' if row['pred_label'] == 0 else 'FAKE'
    print(f'  [{true_str}→{pred_str}] prob={row["prob"]:.3f} conf={row["confidence"]:.3f} | {row["source"]} | {row["topic"]}')
    print(f'    Caption: {row["caption"]}')
    print()

Val samples  : 1000
Correct      : 883 (88.3%)
Errors       : 117 (11.7%)
  FP (real→fake): 71
  FN (fake→real): 46

ERROR RATE BY NEWS SOURCE
                 total  errors  accuracy  error_rate
source                                              
bbc                139      25  0.820144    0.179856
washington_post    157      24  0.847134    0.152866
guardian           474      48  0.898734    0.101266
usa_today          230      20  0.913043    0.086957

ERROR RATE BY TOPIC
                         total  errors  accuracy
topic                                           
australia-news               8       2  0.750000
media                       19       4  0.789474
lifeandstyle                 5       1  0.800000
artanddesign                17       3  0.823529
education                    6       1  0.833333
conflict_attack             18       3  0.833333
uk-news                     12       2  0.833333
stage                       13       2  0.846154
international_relations     

In [37]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
import json, os, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT  = r'D:\Pics Can Lie'
DATASET_ROOT  = os.path.join(PROJECT_ROOT, 'kaggle_dataset_full')
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load signals ──
id_df       = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels      = id_df['label'].values

clip_probs  = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_probs.npy'))
clip_sims   = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_sims.npy'))

deb_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'deberta_val_scores_v2.csv'))
deb_df['id'] = deb_df['id'].astype(str)
deb_lookup   = dict(zip(deb_df['id'], deb_df['entailment_score']))
deb_scores   = np.array([deb_lookup.get(i, 0.33) for i in id_df['id']])

ev_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'evidence_clip_scores.csv'))
ev_df['id'] = ev_df['id'].astype(str)
ev_lookup   = {row['id']: row for _, row in ev_df.iterrows()}
s2 = np.array([ev_lookup.get(i, {}).get('s2', 0.0) for i in id_df['id']])
s3 = np.array([ev_lookup.get(i, {}).get('s3', 0.0) for i in id_df['id']])
s4 = np.array([ev_lookup.get(i, {}).get('s4', 0.0) for i in id_df['id']])
s5 = np.array([ev_lookup.get(i, {}).get('s5', 0.0) for i in id_df['id']])
s6 = np.array([ev_lookup.get(i, {}).get('s6', 0.0) for i in id_df['id']])

wiki_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'wiki_nli_scores.csv'))
wiki_df['id'] = wiki_df['id'].astype(str)
wiki_lookup   = {row['id']: row for _, row in wiki_df.iterrows()}
w1 = np.array([wiki_lookup.get(i, {}).get('wiki_score', 0.33) for i in id_df['id']])

X = np.column_stack([clip_probs, clip_sims, deb_scores, s2, s3, s4, s5, s6, w1])

# ── Load metadata for source info ──
with open(os.path.join(DATASET_ROOT, 'metadata', 'val.json'), 'r') as f:
    metadata = json.load(f)

# ── Load AITR ──
class AITR(nn.Module):
    def __init__(self, embed_dim=768, scalar_dim=9, num_heads=8, num_layers=2,
                 dropout=0.3, hidden_dim=256):
        super().__init__()
        self.scalar_proj    = nn.Sequential(nn.Linear(scalar_dim, embed_dim), nn.LayerNorm(embed_dim))
        self.type_embedding = nn.Embedding(5, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*2,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.classifier  = nn.Sequential(
            nn.LayerNorm(embed_dim), nn.Linear(embed_dim, hidden_dim),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
    def forward(self, img_emb, txt_emb, scalar):
        B = img_emb.size(0)
        tokens   = torch.stack([img_emb, txt_emb, img_emb*txt_emb,
                                  img_emb-txt_emb, self.scalar_proj(scalar)], dim=1)
        type_ids = torch.arange(5, device=tokens.device).unsqueeze(0).expand(B, -1)
        tokens   = tokens + self.type_embedding(type_ids)
        out      = self.transformer(torch.cat([self.cls_token.expand(B,-1,-1), tokens], dim=1))
        return self.classifier(out[:, 0]).squeeze(-1)

img_feats     = F.normalize(torch.load(os.path.join(CLIP_FEATURES, 'clip_img_features.pt')), dim=-1)
txt_feats     = F.normalize(torch.load(os.path.join(CLIP_FEATURES, 'clip_txt_features.pt')), dim=-1)
scaler        = StandardScaler()
scalar_scaled = torch.tensor(scaler.fit_transform(X), dtype=torch.float32)

aitr_model = AITR().to(device)
aitr_model.load_state_dict(torch.load(
    os.path.join(PROJECT_ROOT, 'fusion_aitr', 'aitr_weights.pt'), map_location=device))
aitr_model.eval()

indices = np.arange(len(labels))
_, val_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels)

val_ds = TensorDataset(
    img_feats[val_idx].to(device), txt_feats[val_idx].to(device),
    scalar_scaled[val_idx].to(device),
    torch.tensor(labels[val_idx], dtype=torch.float32)
)
val_ld = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

all_probs, all_labels_val = [], []
with torch.no_grad():
    for img, txt, scl, yb in val_ld:
        logits = aitr_model(img, txt, scl)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels_val.extend(yb.long().numpy())

all_probs      = np.array(all_probs)
all_labels_val = np.array(all_labels_val)
val_ids        = id_df['id'].values[val_idx]

# ── Get per-sample metadata ──
sources     = np.array([metadata.get(sid, {}).get('source', 'unknown') for sid in val_ids])
val_clip_p  = clip_probs[val_idx]
val_clip_s  = clip_sims[val_idx]

# ── Baseline: global threshold 0.54 ──
baseline_preds = (all_probs > 0.54).astype(int)
baseline_acc   = accuracy_score(all_labels_val, baseline_preds)
print(f'Baseline (threshold=0.54): {baseline_acc*100:.2f}%')

# ── Fix 1: Source-aware thresholds ──
# Tuned based on error analysis: BBC hardest, USA Today easiest
source_thresholds = {
    'bbc'            : 0.60,
    'washington_post': 0.57,
    'guardian'       : 0.54,
    'usa_today'      : 0.51,
}
default_threshold = 0.54

fix1_preds = np.zeros(len(all_probs), dtype=int)
for i, (prob, source) in enumerate(zip(all_probs, sources)):
    thr = source_thresholds.get(source, default_threshold)
    fix1_preds[i] = int(prob > thr)

fix1_acc = accuracy_score(all_labels_val, fix1_preds)
fix1_f1  = f1_score(all_labels_val, fix1_preds)
print(f'Fix 1 (source-aware threshold): {fix1_acc*100:.2f}%  F1={fix1_f1:.4f}')

# ── Fix 2: High-confidence gate for FP reduction ──
# If CLIP is very confident it's real, override AITR
# Condition: clip_prob < 0.25 AND clip_sim < -0.02 → force REAL
fix2_preds = baseline_preds.copy()
gate_mask  = (val_clip_p < 0.25) & (val_clip_s < -0.02)
fix2_preds[gate_mask] = 0  # override to REAL
fix2_acc = accuracy_score(all_labels_val, fix2_preds)
fix2_f1  = f1_score(all_labels_val, fix2_preds)
print(f'Fix 2 (FP gate):               {fix2_acc*100:.2f}%  F1={fix2_f1:.4f}')
print(f'  Samples gated to REAL: {gate_mask.sum()}')

# ── Fix 1 + Fix 2 combined ──
combined_preds = fix1_preds.copy()
combined_preds[gate_mask] = 0
combined_acc = accuracy_score(all_labels_val, combined_preds)
combined_f1  = f1_score(all_labels_val, combined_preds)
print(f'Fix 1 + Fix 2 combined:        {combined_acc*100:.2f}%  F1={combined_f1:.4f}')

# ── Fine-tune source thresholds with grid search ──
print()
print('=' * 60)
print('GRID SEARCH — optimal source thresholds')
print('=' * 60)

unique_sources = list(source_thresholds.keys())
best_acc_grid  = fix1_acc
best_thresholds = source_thresholds.copy()

# Search ±0.08 around initial thresholds
for src in unique_sources:
    src_mask   = sources == src
    if src_mask.sum() < 10:
        continue
    init_thr   = source_thresholds[src]
    best_src_acc = 0
    best_src_thr = init_thr
    for thr in np.arange(init_thr - 0.08, init_thr + 0.09, 0.01):
        test_preds = fix1_preds.copy()
        test_preds[src_mask] = (all_probs[src_mask] > thr).astype(int)
        acc = accuracy_score(all_labels_val, test_preds)
        if acc > best_src_acc:
            best_src_acc = acc
            best_src_thr = thr
    best_thresholds[src] = round(best_src_thr, 2)
    print(f'  {src:<20}: {init_thr:.2f} → {best_src_thr:.2f}  (acc={best_src_acc*100:.2f}%)')

# Apply optimized thresholds
opt_preds = np.zeros(len(all_probs), dtype=int)
for i, (prob, source) in enumerate(zip(all_probs, sources)):
    thr = best_thresholds.get(source, default_threshold)
    opt_preds[i] = int(prob > thr)
opt_acc = accuracy_score(all_labels_val, opt_preds)
opt_f1  = f1_score(all_labels_val, opt_preds)

print()
print('=' * 60)
print('FINAL RESULTS')
print('=' * 60)
results = [
    ('Baseline (global thr=0.54)',        baseline_acc * 100),
    ('Fix 1 (source-aware thresholds)',   fix1_acc * 100),
    ('Fix 2 (FP gate)',                   fix2_acc * 100),
    ('Fix 1 + Fix 2',                     combined_acc * 100),
    ('Optimized source thresholds',       opt_acc * 100),
]
for name, acc in results:
    marker = ' <-- best' if acc == max(r[1] for r in results) else ''
    print(f'  {name:<40} {acc:.2f}%{marker}')

print()
print(f'Best vs previous best (88.30%): {max(r[1] for r in results) - 88.30:+.2f}%')

# ── Classification report for best result ──
best_preds = opt_preds if opt_acc >= combined_acc else combined_preds
print()
print(f'Classification report (best result):')
print(classification_report(all_labels_val, best_preds, target_names=['REAL', 'FAKE']))

Baseline (threshold=0.54): 88.30%
Fix 1 (source-aware threshold): 88.20%  F1=0.8843
Fix 2 (FP gate):               88.30%  F1=0.8859
  Samples gated to REAL: 0
Fix 1 + Fix 2 combined:        88.20%  F1=0.8843

GRID SEARCH — optimal source thresholds
  bbc                 : 0.60 → 0.58  (acc=88.40%)
  washington_post     : 0.57 → 0.57  (acc=88.20%)
  guardian            : 0.54 → 0.54  (acc=88.20%)
  usa_today           : 0.51 → 0.44  (acc=88.30%)

FINAL RESULTS
  Baseline (global thr=0.54)               88.30%
  Fix 1 (source-aware thresholds)          88.20%
  Fix 2 (FP gate)                          88.30%
  Fix 1 + Fix 2                            88.20%
  Optimized source thresholds              88.50% <-- best

Best vs previous best (88.30%): +0.20%

Classification report (best result):
              precision    recall  f1-score   support

        REAL       0.91      0.86      0.88       500
        FAKE       0.87      0.91      0.89       500

    accuracy                      

In [2]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import json
import pandas as pd
import os

# Load StreetCLIP
print('Loading StreetCLIP...')
model     = CLIPModel.from_pretrained('geolocal/StreetCLIP')
processor = CLIPProcessor.from_pretrained('geolocal/StreetCLIP')
model.eval()

PROJECT_ROOT = r'D:\Pics Can Lie'
DATASET_ROOT = os.path.join(PROJECT_ROOT, 'kaggle_dataset_full')

with open(os.path.join(DATASET_ROOT, 'metadata', 'val.json')) as f:
    metadata = json.load(f)

id_df = pd.read_csv(r'D:\Pics Can Lie\clip_finetuned_v2\val_features\val_sample_ids.csv')
id_df['id'] = id_df['id'].astype(str)

def resolve_image_path(rel_path):
    return os.path.join(DATASET_ROOT, str(rel_path).replace('visual_news/', 'images/'))

# Test on first 50 samples with GPE entities
results = []
tested  = 0

for sid in id_df['id'].values:
    if tested >= 50:
        break
    meta     = metadata.get(sid, {})
    entities = meta.get('caption_entities_rel', [])
    gpe      = [e for e in entities
                if isinstance(e, list) and len(e) >= 4
                and e[1] in ['LOC', 'GPE'] and float(e[3]) >= 0.75]
    if not gpe:
        continue

    img_path = resolve_image_path(meta.get('image_path', ''))
    if not os.path.exists(img_path):
        continue

    # Get location names from caption entities
    loc_names = [e[2] for e in gpe[:3]]  # surface form e.g. "London", "Paris"

    try:
        img    = Image.open(img_path).convert('RGB')
        inputs = processor(
            text=loc_names + ['somewhere else'],
            images=img,
            return_tensors='pt', padding=True
        )
        with torch.no_grad():
            logits = model(**inputs).logits_per_image
            probs  = logits.softmax(dim=1).squeeze()

        top_idx  = probs.argmax().item()
        top_loc  = (loc_names + ['somewhere else'])[top_idx]
        top_prob = probs[top_idx].item()

        results.append({
            'id'        : sid,
            'caption'   : meta.get('caption', '')[:80],
            'gpe_locs'  : loc_names,
            'top_pred'  : top_loc,
            'top_prob'  : top_prob,
            'correct'   : top_loc in loc_names,  # did it pick a caption location?
        })
        tested += 1

    except Exception as e:
        continue

# Show results
correct = sum(r['correct'] for r in results)
print(f'\nTested: {tested} samples')
print(f'Correct (picked caption location): {correct}/{tested} ({correct/tested*100:.1f}%)')
print()
for r in results[:15]:
    tick = '✓' if r['correct'] else '✗'
    print(f'{tick} Pred: {r["top_pred"]:<20} (p={r["top_prob"]:.2f}) | Locs: {r["gpe_locs"]} | {r["caption"][:60]}')

d:\Pics Can Lie\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading StreetCLIP...


d:\Pics Can Lie\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Youssef Elghandour\.cache\huggingface\hub\models--geolocal--StreetCLIP. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 590/590 [00:00<00:00, 15948.85it/s]
CLIPModel LOAD REPORT from: geolocal/StreetCL


Tested: 50 samples
Correct (picked caption location): 49/50 (98.0%)

✓ Pred: Phillipsburg         (p=0.75) | Locs: ['Phillipsburg', 'South Main Street'] | Paul Fix of Phillipsburg walks along South Main Street as sn
✓ Pred: New York             (p=0.99) | Locs: ['New York'] | The New York mayor Michael Bloomberg said the search would p
✓ Pred: Kobani               (p=1.00) | Locs: ['Kobani'] | Sanliurfa Turkey Turkish residents watch the clashes between
✓ Pred: Idlib                (p=1.00) | Locs: ['Idlib'] | Two boys whose family fled its home in Idlib walk to their t
✓ Pred: Amsterdam            (p=1.00) | Locs: ['Amsterdam'] | Beach volleyball players in action during a women s poule ma
✓ Pred: Washington           (p=0.65) | Locs: ['Washington'] | The area where bodies regularly receive autopsies at the Dep
✓ Pred: Brazil               (p=1.00) | Locs: ['Brazil', 'Mexico'] | Neymar reacts during Brazil s game against Mexico
✓ Pred: Sri Lanka            (p=0.83) | Locs: ['Bandaran

In [5]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import torch.nn.functional as F
import json
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = r'D:\Pics Can Lie'
DATASET_ROOT = os.path.join(PROJECT_ROOT, 'kaggle_dataset_full')
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')
GEO_CACHE    = os.path.join(PROJECT_ROOT, 'geo_scores.csv')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Load StreetCLIP ──
print('Loading StreetCLIP...')
geo_model     = CLIPModel.from_pretrained('geolocal/StreetCLIP').to(device).eval()
geo_processor = CLIPProcessor.from_pretrained('geolocal/StreetCLIP')
print('StreetCLIP loaded.')

# ── Load data ──
id_df = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels = id_df['label'].values

with open(os.path.join(DATASET_ROOT, 'metadata', 'val.json'), 'r') as f:
    metadata = json.load(f)

def resolve_image_path(rel_path):
    return os.path.join(DATASET_ROOT, str(rel_path).replace('visual_news/', 'images/'))

def get_geo_score(image_pil, loc_names):
    """
    Returns:
      geo_match_prob  — probability that image matches caption locations (0-1)
      geo_max_prob    — max probability assigned to any caption location
      geo_top_in_cap  — 1 if top prediction is a caption location, 0 otherwise
      n_locs          — number of location entities found
    """
    candidates = loc_names[:4] + ['somewhere else']  # max 4 locations + fallback
    try:
        inputs = geo_processor(
            text=candidates, images=image_pil,
            return_tensors='pt', padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = geo_model(**inputs).logits_per_image
            probs  = logits.softmax(dim=1).squeeze()

        # Probability mass on caption locations (excludes "somewhere else")
        cap_probs    = probs[:-1]
        geo_match    = cap_probs.sum().item()
        geo_max      = cap_probs.max().item()
        top_idx      = probs.argmax().item()
        top_in_cap   = float(top_idx < len(loc_names))

        return geo_match, geo_max, top_in_cap, len(loc_names)
    except:
        return 0.5, 0.5, 0.5, 0

# ── Resume logic ──
if os.path.exists(GEO_CACHE):
    geo_df       = pd.read_csv(GEO_CACHE)
    geo_df['id'] = geo_df['id'].astype(str)
    already_done = set(geo_df['id'].values)
    print(f'Resuming — {len(already_done)} already scored')
else:
    geo_df       = pd.DataFrame()
    already_done = set()
    print('Starting fresh')

remaining   = [sid for sid in id_df['id'].values if sid not in already_done]
new_results = []
no_gpe      = 0
errors      = 0
SAVE_EVERY  = 100

print(f'Samples to score: {len(remaining)}')
print('Saves every 100 samples — safe to interrupt and resume.')

for i, sample_id in enumerate(tqdm(remaining, desc='GeoScore')):
    try:
        meta     = metadata.get(sample_id, {})
        entities = meta.get('caption_entities_rel', [])

        # Extract GPE/LOC entities with confidence >= 0.75
        gpe = [e for e in entities
               if isinstance(e, list) and len(e) >= 4
               and e[1] in ['LOC', 'GPE'] and float(e[3]) >= 0.75]

        if not gpe:
            # No location entities — neutral score
            no_gpe += 1
            new_results.append({
                'id'           : sample_id,
                'geo_match'    : 0.5,
                'geo_max'      : 0.5,
                'geo_top_in_cap': 0.5,
                'n_locs'       : 0,
                'has_gpe'      : 0,
            })
            continue

        loc_names = [e[2] for e in gpe[:4]]  # surface forms e.g. "London"
        img_path  = resolve_image_path(meta.get('image_path', ''))
        img       = Image.open(img_path).convert('RGB')

        geo_match, geo_max, top_in_cap, n_locs = get_geo_score(img, loc_names)

        new_results.append({
            'id'            : sample_id,
            'geo_match'     : geo_match,
            'geo_max'       : geo_max,
            'geo_top_in_cap': top_in_cap,
            'n_locs'        : n_locs,
            'has_gpe'       : 1,
        })

    except Exception as e:
        errors += 1
        new_results.append({
            'id'            : sample_id,
            'geo_match'     : 0.5,
            'geo_max'       : 0.5,
            'geo_top_in_cap': 0.5,
            'n_locs'        : 0,
            'has_gpe'       : 0,
        })

    # Save every 100 samples
    if len(new_results) % SAVE_EVERY == 0:
        partial  = pd.DataFrame(new_results)
        combined = pd.concat([geo_df, partial], ignore_index=True) if not geo_df.empty else partial
        combined.to_csv(GEO_CACHE, index=False)

# Final save
final_geo = pd.DataFrame(new_results)
if not geo_df.empty:
    final_geo = pd.concat([geo_df, final_geo], ignore_index=True)
final_geo.to_csv(GEO_CACHE, index=False)

print(f'\nDone.')
print(f'  Scored   : {len(final_geo)}')
print(f'  No GPE   : {no_gpe} ({no_gpe/len(final_geo)*100:.1f}%)')
print(f'  Errors   : {errors}')
print(f'  With GPE : {len(final_geo) - no_gpe} ({(len(final_geo)-no_gpe)/len(final_geo)*100:.1f}%)')

# ── Quick discriminability check ──
merged = final_geo.merge(id_df[['id','label']], on='id')
print()
print('=' * 55)
print('SIGNAL DISCRIMINABILITY')
print('=' * 55)
for col in ['geo_match', 'geo_max', 'geo_top_in_cap']:
    real = merged[col][merged['label'] == 0]
    fake = merged[col][merged['label'] == 1]
    print(f'{col}:')
    print(f'  real: {real.mean():.4f}  fake: {fake.mean():.4f}  diff: {fake.mean()-real.mean():+.4f}')

# GPE samples only
gpe_only = merged[merged['has_gpe'] == 1]
print(f'\nGPE samples only ({len(gpe_only)} samples):')
for col in ['geo_match', 'geo_max', 'geo_top_in_cap']:
    real = gpe_only[col][gpe_only['label'] == 0]
    fake = gpe_only[col][gpe_only['label'] == 1]
    print(f'{col}:')
    print(f'  real: {real.mean():.4f}  fake: {fake.mean():.4f}  diff: {fake.mean()-real.mean():+.4f}')

Device: cuda
Loading StreetCLIP...


Loading weights: 100%|██████████| 590/590 [00:00<00:00, 9672.38it/s]
CLIPModel LOAD REPORT from: geolocal/StreetCLIP
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


StreetCLIP loaded.
Starting fresh
Samples to score: 5000
Saves every 100 samples — safe to interrupt and resume.


GeoScore: 100%|██████████| 5000/5000 [08:27<00:00,  9.86it/s]


Done.
  Scored   : 5000
  No GPE   : 2522 (50.4%)
  Errors   : 0
  With GPE : 2478 (49.6%)

SIGNAL DISCRIMINABILITY
geo_match:
  real: 0.6970  fake: 0.6989  diff: +0.0018
geo_max:
  real: 0.6748  fake: 0.6768  diff: +0.0019
geo_top_in_cap:
  real: 0.7116  fake: 0.7137  diff: +0.0021

GPE samples only (4234 samples):
geo_match:
  real: 0.8974  fake: 0.9008  diff: +0.0033
geo_max:
  real: 0.8526  fake: 0.8562  diff: +0.0036
geo_top_in_cap:
  real: 0.9267  fake: 0.9306  diff: +0.0038


In [12]:
from groundingdino.util.inference import load_model, load_image, predict
import torch
import torch.nn.functional as F
import clip as clip_lib
from PIL import Image
import numpy as np
import json
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT  = r'D:\Pics Can Lie'
DATASET_ROOT  = os.path.join(PROJECT_ROOT, 'kaggle_dataset_full')
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')
CLIP_MODEL_DIR = os.path.join(PROJECT_ROOT, 'models', 'clip')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load Grounding DINO ──
print('Loading Grounding DINO...')
from huggingface_hub import hf_hub_download

# Download config and weights if not cached
config_path = hf_hub_download(
    repo_id='ShilongLiu/GroundingDINO',
    filename='GroundingDINO_SwinT_OGC.cfg.py'
)
weights_path = hf_hub_download(
    repo_id='ShilongLiu/GroundingDINO',
    filename='groundingdino_swint_ogc.pth'
)

gdino = load_model(config_path, weights_path, device=str(device))
gdino.eval()
print('Grounding DINO loaded.')

# ── Load CLIP for crop scoring ──
os.environ['CLIP_DOWNLOAD_ROOT'] = CLIP_MODEL_DIR
print('Loading CLIP...')
clip_model, clip_preprocess = clip_lib.load('ViT-L/14', device=device, jit=False)
clip_model = clip_model.float().eval()
print('CLIP loaded.')

# ── Load data ──
id_df = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels = id_df['label'].values

with open(os.path.join(DATASET_ROOT, 'metadata', 'val.json'), 'r') as f:
    metadata = json.load(f)

def resolve_image_path(rel_path):
    return os.path.join(DATASET_ROOT, str(rel_path).replace('visual_news/', 'images/'))

def get_entity_clip_score(image_pil, entity_text, boxes, logits):
    """
    For each detected bounding box, crop the image and compute
    CLIP similarity between the crop and the entity text.
    Returns max similarity across all detected boxes.
    """
    if len(boxes) == 0:
        return None  # entity not detected

    W, H = image_pil.size
    scores = []

    for box in boxes:
        # box is [cx, cy, w, h] normalized
        cx, cy, bw, bh = box.tolist()
        x1 = max(0, int((cx - bw/2) * W))
        y1 = max(0, int((cy - bh/2) * H))
        x2 = min(W, int((cx + bw/2) * W))
        y2 = min(H, int((cy + bh/2) * H))

        if x2 - x1 < 10 or y2 - y1 < 10:
            continue

        crop = image_pil.crop((x1, y1, x2, y2))

        with torch.no_grad():
            img_tensor = clip_preprocess(crop).unsqueeze(0).to(device)
            txt_tokens  = clip_lib.tokenize([entity_text], truncate=True).to(device)
            img_feat    = F.normalize(clip_model.encode_image(img_tensor).float(), dim=-1)
            txt_feat    = F.normalize(clip_model.encode_text(txt_tokens).float(), dim=-1)
            sim         = (img_feat * txt_feat).sum().item()
        scores.append(sim)

    return max(scores) if scores else None

# ── Test on 20 samples with PERSON entities ──
print('\nTesting on 20 samples with PERSON entities...')
print('=' * 70)

tested = 0
results = []

for sid in id_df['id'].values:
    if tested >= 20:
        break

    meta     = metadata.get(sid, {})
    entities = meta.get('caption_entities_rel', [])

    # Focus on PERSON entities — most discriminative for OOC detection
    persons = [e for e in entities
               if isinstance(e, list) and len(e) >= 4
               and e[1] == 'PER' and float(e[3]) >= 0.80]

    if not persons or not metadata.get(sid, {}).get('image_has_person', False):
        continue

    img_path = resolve_image_path(meta.get('image_path', ''))
    if not os.path.exists(img_path):
        continue

    try:
        image_pil = Image.open(img_path).convert('RGB')
        image_source, image_tensor = load_image(img_path)

        entity_scores = []
        for ent in persons[:2]:  # test top 2 persons
            entity_name = ent[2]  # surface form e.g. "Barack Obama"
            wiki_name   = ent[0]  # Wikipedia name e.g. "Barack_Obama"

            # Grounding DINO: find entity in image
            boxes, logits, phrases = predict(
                model     = gdino,
                image     = image_tensor,
                caption   = entity_name,
                box_threshold  = 0.25,
                text_threshold = 0.20,
                device    = str(device),
            )

            n_detections = len(boxes)

            # CLIP score on detected crops
            clip_score = get_entity_clip_score(image_pil, entity_name, boxes, logits)

            entity_scores.append({
                'entity'      : entity_name,
                'n_detected'  : n_detections,
                'clip_score'  : clip_score,
                'gdino_conf'  : logits.max().item() if len(logits) > 0 else 0.0,
            })

        label_str = 'REAL' if labels[list(id_df['id'].values).index(sid)] == 0 else 'FAKE'
        caption   = meta.get('caption', '')[:80]

        print(f'[{label_str}] {caption}')
        for es in entity_scores:
            detected = '✓' if es['n_detected'] > 0 else '✗'
            clip_str = f'{es["clip_score"]:.3f}' if es['clip_score'] is not None else 'N/A'
            print(f'  {detected} {es["entity"]:<25} detections={es["n_detected"]} gdino_conf={es["gdino_conf"]:.3f} clip_sim={clip_str}')
        print()

        results.append({
            'id'          : sid,
            'label'       : label_str,
            'n_persons'   : len(persons),
            'any_detected': any(es['n_detected'] > 0 for es in entity_scores),
            'mean_clip'   : np.mean([es['clip_score'] for es in entity_scores
                                     if es['clip_score'] is not None]) if entity_scores else None,
        })
        tested += 1

    except Exception as e:
        print(f'ERROR on {sid}: {e}')
        continue

# Summary
detected_count = sum(1 for r in results if r['any_detected'])
print('=' * 70)
print(f'SUMMARY: {tested} samples tested')
print(f'  Entity detected in image: {detected_count}/{tested} ({detected_count/max(tested,1)*100:.0f}%)')
print(f'  Mean CLIP score (when detected): '
      f'{np.mean([r["mean_clip"] for r in results if r["mean_clip"] is not None]):.3f}')
print()
print('If detection rate > 50% and CLIP scores > 0.20 → worth running full pipeline')
print('If detection rate < 30% or CLIP scores < 0.15 → signal will be noise')

Loading Grounding DINO...
final text_encoder_type: bert-base-uncased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5802.78it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AttributeError: 'BertModel' object has no attribute 'get_head_mask'

In [3]:
import json
import pandas as pd
import numpy as np

# ── Full ablation results ──
with open(r'D:\Pics Can Lie\ablation_ateeq_vs_sightengine.json') as f:
    ablation = json.load(f)
print(json.dumps(ablation, indent=2))

print()

# ── Ateeq before vs after fine-tuning on MMFakeBench 200 samples ──
df = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_ai_scores_finetuned.csv')

print('=== ATEEQ BEFORE FINE-TUNING ===')
# Before: predicted_label == 'ai' means predicted fake
df['true_label'] = (df['gt_answers'] == 'Fake').astype(int)
df['pred_before'] = (df['predicted_label'] == 'ai').astype(int)
df['pred_after']  = (df['predicted_label_ft'] == 'ai').astype(int)

from sklearn.metrics import accuracy_score, f1_score, classification_report

print(f'Overall accuracy : {accuracy_score(df["true_label"], df["pred_before"])*100:.2f}%')
print(f'Overall F1       : {f1_score(df["true_label"], df["pred_before"]):.4f}')
print()
print('By fake_cls:')
for cat in df['fake_cls'].unique():
    mask = df['fake_cls'] == cat
    acc  = accuracy_score(df['true_label'][mask], df['pred_before'][mask])
    print(f'  {cat:<35} {mask.sum():>3} samples | acc={acc*100:.1f}%')

print()
print('=== ATEEQ AFTER FINE-TUNING ===')
print(f'Overall accuracy : {accuracy_score(df["true_label"], df["pred_after"])*100:.2f}%')
print(f'Overall F1       : {f1_score(df["true_label"], df["pred_after"]):.4f}')
print()
print('By fake_cls:')
for cat in df['fake_cls'].unique():
    mask = df['fake_cls'] == cat
    acc  = accuracy_score(df['true_label'][mask], df['pred_after'][mask])
    print(f'  {cat:<35} {mask.sum():>3} samples | acc={acc*100:.1f}%')

print()
# ── Fusion features 300 ──
df_fusion = pd.read_csv(r'D:\Pics Can Lie\fusion_features_300_ateeq.csv')
print('=== FUSION FEATURES (300 samples) ===')
print(f'Columns: {df_fusion.columns.tolist()}')
print(f'Label dist: real={( df_fusion["label"]==0).sum()} fake={(df_fusion["label"]==1).sum()}')
print(f'Ateeq score stats: mean={df_fusion["ateeq_score"].mean():.3f} std={df_fusion["ateeq_score"].std():.3f}')
real_mean = df_fusion["ateeq_score"][df_fusion["label"]==0].mean()
fake_mean = df_fusion["ateeq_score"][df_fusion["label"]==1].mean()
print(f'Real mean: {real_mean:.3f} | Fake mean: {fake_mean:.3f} | Diff: {fake_mean-real_mean:+.3f}')

{
  "description": "Ablation comparing SightEngine vs Ateeqq as AI module, n=300, 5-fold CV",
  "n_samples": 300,
  "results": [
    {
      "combination": "BLIP only",
      "accuracy": 0.8267,
      "f1": 0.8256,
      "precision": 0.8327,
      "recall": 0.82
    },
    {
      "combination": "DeBERTa only",
      "accuracy": 0.58,
      "f1": 0.6795,
      "precision": 0.5523,
      "recall": 0.8867
    },
    {
      "combination": "BLIP + SightEngine",
      "accuracy": 0.88,
      "f1": 0.8809,
      "precision": 0.877,
      "recall": 0.8867
    },
    {
      "combination": "BLIP + Ateeqq",
      "accuracy": 0.8233,
      "f1": 0.8227,
      "precision": 0.8265,
      "recall": 0.82
    },
    {
      "combination": "BLIP + DeBERTa",
      "accuracy": 0.85,
      "f1": 0.8484,
      "precision": 0.8581,
      "recall": 0.84
    },
    {
      "combination": "BLIP + DeBERTa + SightEngine",
      "accuracy": 0.8967,
      "f1": 0.8958,
      "precision": 0.9013,
      "recall": 

In [4]:
import json
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# ── Full ablation ──
with open(r'D:\Pics Can Lie\ablation_ateeq_vs_sightengine.json') as f:
    ablation = json.load(f)

print('ALL ABLATION COMBINATIONS:')
for r in ablation['results']:
    print(f'  {r["combination"]:<40} acc={r["accuracy"]*100:.2f}%  f1={r["f1"]:.4f}')

print()

# ── MMFakeBench Ateeq before/after ──
df = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_ai_scores_finetuned.csv')
df['true_label'] = (df['gt_answers'] == 'Fake').astype(int)
df['pred_before'] = (df['predicted_label'] == 'ai').astype(int)
df['pred_after']  = (df['predicted_label_ft'] == 'ai').astype(int)

print('ATEEQ BEFORE fine-tuning — MMFakeBench (200 samples):')
print(f'  Accuracy: {accuracy_score(df["true_label"], df["pred_before"])*100:.2f}%')
print(f'  F1      : {f1_score(df["true_label"], df["pred_before"], zero_division=0):.4f}')
for cat in sorted(df['fake_cls'].unique()):
    mask = df['fake_cls'] == cat
    acc  = accuracy_score(df['true_label'][mask], df['pred_before'][mask])
    n    = mask.sum()
    print(f'    {cat:<35} n={n:>3}  acc={acc*100:.1f}%')

print()
print('ATEEQ AFTER fine-tuning — MMFakeBench (200 samples):')
print(f'  Accuracy: {accuracy_score(df["true_label"], df["pred_after"])*100:.2f}%')
print(f'  F1      : {f1_score(df["true_label"], df["pred_after"], zero_division=0):.4f}')
for cat in sorted(df['fake_cls'].unique()):
    mask = df['fake_cls'] == cat
    acc  = accuracy_score(df['true_label'][mask], df['pred_after'][mask])
    n    = mask.sum()
    print(f'    {cat:<35} n={n:>3}  acc={acc*100:.1f}%')

print()
print('ATEEQ on NewsCLIPpings (300 samples):')
print(f'  Real mean: 0.407 | Fake mean: 0.438 | Diff: +0.030')
print(f'  This is weak — confirms Ateeq adds noise on NewsCLIPpings real photos')

ALL ABLATION COMBINATIONS:
  BLIP only                                acc=82.67%  f1=0.8256
  DeBERTa only                             acc=58.00%  f1=0.6795
  BLIP + SightEngine                       acc=88.00%  f1=0.8809
  BLIP + Ateeqq                            acc=82.33%  f1=0.8227
  BLIP + DeBERTa                           acc=85.00%  f1=0.8484
  BLIP + DeBERTa + SightEngine             acc=89.67%  f1=0.8958
  BLIP + DeBERTa + Ateeqq                  acc=85.33%  f1=0.8512

ATEEQ BEFORE fine-tuning — MMFakeBench (200 samples):
  Accuracy: 56.00%
  F1      : 0.5217
    original                            n=100  acc=64.0%
    visual_veracity_distortion          n=100  acc=48.0%

ATEEQ AFTER fine-tuning — MMFakeBench (200 samples):
  Accuracy: 75.50%
  F1      : 0.7803
    original                            n=100  acc=64.0%
    visual_veracity_distortion          n=100  acc=87.0%

ATEEQ on NewsCLIPpings (300 samples):
  Real mean: 0.407 | Fake mean: 0.438 | Diff: +0.030
  This is wea

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

df = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_ai_scores_finetuned.csv')
df['true_label'] = (df['gt_answers'] == 'Fake').astype(int)
df['pred_ft']    = (df['predicted_label_ft'] == 'ai').astype(int)

# ── Overall on 200 samples ──
print('=' * 65)
print('FINE-TUNED ATEEQ — MMFakeBench (200 samples)')
print('=' * 65)
print(f'Accuracy : {accuracy_score(df["true_label"], df["pred_ft"])*100:.2f}%')
print(f'F1       : {f1_score(df["true_label"], df["pred_ft"]):.4f}')
print()
print(classification_report(df['true_label'], df['pred_ft'],
                              target_names=['REAL', 'FAKE']))

# ── Per category ──
print('Per category:')
for cat in sorted(df['fake_cls'].unique()):
    mask = df['fake_cls'] == cat
    acc  = accuracy_score(df['true_label'][mask], df['pred_ft'][mask])
    f1   = f1_score(df['true_label'][mask], df['pred_ft'][mask], zero_division=0)
    n    = mask.sum()
    print(f'  {cat:<35} n={n:>3}  acc={acc*100:.1f}%  f1={f1:.4f}')

# ── Threshold sweep on ai_score_ft ──
print()
print('Threshold sweep on ai_score_ft:')
best_acc, best_thr = 0, 0.5
for thr in np.arange(0.20, 0.81, 0.05):
    preds = (df['ai_score_ft'] > thr).astype(int)
    acc   = accuracy_score(df['true_label'], preds)
    f1    = f1_score(df['true_label'], preds, zero_division=0)
    if acc > best_acc:
        best_acc, best_thr = acc, thr
    print(f'  thr={thr:.2f}  acc={acc*100:.2f}%  f1={f1:.4f}')

print(f'\nBest: {best_acc*100:.2f}% at threshold {best_thr:.2f}')

# ── Visual veracity distortion only (what Ateeq was trained for) ──
print()
print('Visual veracity distortion only (100 samples):')
vv = df[df['fake_cls'] == 'visual_veracity_distortion']
best_vv_acc, best_vv_thr = 0, 0.5
for thr in np.arange(0.20, 0.81, 0.05):
    preds = (vv['ai_score_ft'] > thr).astype(int)
    acc   = accuracy_score(vv['true_label'], preds)
    if acc > best_vv_acc:
        best_vv_acc, best_vv_thr = acc, thr
print(f'  Best: {best_vv_acc*100:.2f}% at threshold {best_vv_thr:.2f}')

FINE-TUNED ATEEQ — MMFakeBench (200 samples)
Accuracy : 75.50%
F1       : 0.7803

              precision    recall  f1-score   support

        REAL       0.83      0.64      0.72       100
        FAKE       0.71      0.87      0.78       100

    accuracy                           0.76       200
   macro avg       0.77      0.76      0.75       200
weighted avg       0.77      0.76      0.75       200

Per category:
  original                            n=100  acc=64.0%  f1=0.0000
  visual_veracity_distortion          n=100  acc=87.0%  f1=0.9305

Threshold sweep on ai_score_ft:
  thr=0.20  acc=59.50%  f1=0.6989
  thr=0.25  acc=62.00%  f1=0.7121
  thr=0.30  acc=66.50%  f1=0.7352
  thr=0.35  acc=71.50%  f1=0.7654
  thr=0.40  acc=73.50%  f1=0.7782
  thr=0.45  acc=72.50%  f1=0.7639
  thr=0.50  acc=75.50%  f1=0.7803
  thr=0.55  acc=77.50%  f1=0.7887
  thr=0.60  acc=79.00%  f1=0.7961
  thr=0.65  acc=79.50%  f1=0.7960
  thr=0.70  acc=79.50%  f1=0.7853
  thr=0.75  acc=80.50%  f1=0.7914
  th

In [3]:
import torch
weights = torch.load(r'D:\Pics Can Lie\fusion_aitr\aitr_weights.pt', map_location='cpu')
print(list(weights.keys()))

['cls_token', 'scalar_proj.0.weight', 'scalar_proj.0.bias', 'scalar_proj.1.weight', 'scalar_proj.1.bias', 'type_embedding.weight', 'transformer.layers.0.self_attn.in_proj_weight', 'transformer.layers.0.self_attn.in_proj_bias', 'transformer.layers.0.self_attn.out_proj.weight', 'transformer.layers.0.self_attn.out_proj.bias', 'transformer.layers.0.linear1.weight', 'transformer.layers.0.linear1.bias', 'transformer.layers.0.linear2.weight', 'transformer.layers.0.linear2.bias', 'transformer.layers.0.norm1.weight', 'transformer.layers.0.norm1.bias', 'transformer.layers.0.norm2.weight', 'transformer.layers.0.norm2.bias', 'transformer.layers.1.self_attn.in_proj_weight', 'transformer.layers.1.self_attn.in_proj_bias', 'transformer.layers.1.self_attn.out_proj.weight', 'transformer.layers.1.self_attn.out_proj.bias', 'transformer.layers.1.linear1.weight', 'transformer.layers.1.linear1.bias', 'transformer.layers.1.linear2.weight', 'transformer.layers.1.linear2.bias', 'transformer.layers.1.norm1.weigh

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'D:\Pics Can Lie\verite\VERITE.csv')
img = np.load(r'D:\Pics Can Lie\verite\VERITE_clip_image_embeddings_ViTL14.npy')

print(f"CSV rows: {len(df)}")
print(f"Image embeddings shape: {img.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")

CSV rows: 1001
Image embeddings shape: (1001, 768)
Label distribution:
label
true              338
miscaptioned      338
out-of-context    325
Name: count, dtype: int64


In [5]:
print("scalar_proj.0.weight shape:", weights['scalar_proj.0.weight'].shape)
print("classifier.0.weight shape:", weights['classifier.0.weight'].shape)
print("classifier.1.weight shape:", weights['classifier.1.weight'].shape)
print("classifier.4.weight shape:", weights['classifier.4.weight'].shape)
print("type_embedding.weight shape:", weights['type_embedding.weight'].shape)
print("cls_token shape:", weights['cls_token'].shape)
print("transformer.layers.0.self_attn.in_proj_weight shape:", weights['transformer.layers.0.self_attn.in_proj_weight'].shape)

scalar_proj.0.weight shape: torch.Size([768, 9])
classifier.0.weight shape: torch.Size([768])
classifier.1.weight shape: torch.Size([256, 768])
classifier.4.weight shape: torch.Size([1, 256])
type_embedding.weight shape: torch.Size([5, 768])
cls_token shape: torch.Size([1, 1, 768])
transformer.layers.0.self_attn.in_proj_weight shape: torch.Size([2304, 768])


In [6]:
import pandas as pd
df = pd.read_csv(r'D:\Pics Can Lie\verite\VERITE_articles.csv')
print(df.columns.tolist())
print(df.head(3))

['Unnamed: 0', 'id', 'true_url', 'false_caption', 'true_caption', 'false_url', 'query', 'snopes_url']
   Unnamed: 0  id                                           true_url  \
0           0   0  https://mediaproxy.snopes.com/width/600/https:...   
1           1   1  https://ic.pics.livejournal.com/elesika73/5218...   
2           2   2  https://mediaproxy.snopes.com/width/1200/https...   

                                       false_caption  \
0  Photograph shows Chinese officials in white pr...   
1  Image shows electric green scooters that have ...   
2  Image shows Balenciaga stylist Lotta Volkova h...   

                                        true_caption  \
0  Photograph shows a family that was taken away ...   
1  Image shows electric scooters abandoned due to...   
2  Image shows unnamed runway model showcasing de...   

                                           false_url  \
0       https://static.dw.com/image/63908432_604.jpg   
1  https://miro.medium.com/max/1400/0*-wsuYdEl6

In [7]:
import torch
from PIL import Image
import requests
from io import BytesIO

# Test one URL loads
url = "https://mediaproxy.snopes.com/width/600/https://www.snopes.com/uploads/2020/02/china-family-covid.jpg"
resp = requests.get(url, timeout=10)
img = Image.open(BytesIO(resp.content)).convert('RGB')
print(f"Image loaded: {img.size}")

# Test CLIP v2 loads
import sys
sys.path.append(r'D:\Pics Can Lie')
import clip
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
clip_model, preprocess = clip.load('ViT-L/14', device=device, jit=False)
clip_model = clip_model.float()
print(f"CLIP loaded on {device}")

# Test classifier loads
checkpoint = torch.load(r'D:\Pics Can Lie\clip_finetuned_v2\clip_classifier.pt', map_location=device)
print(f"Classifier keys: {list(checkpoint.keys())[:5]}")

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x0000017E65F50360>

In [8]:
import requests

url = "https://mediaproxy.snopes.com/width/600/https://www.snopes.com/uploads/2020/02/china-family-covid.jpg"
resp = requests.get(url, timeout=10, headers={
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
})
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type')}")
print(f"Content length: {len(resp.content)}")

# Try the false_url instead (not a proxy)
url2 = "https://static.dw.com/image/63908432_604.jpg"
resp2 = requests.get(url2, timeout=10, headers={
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
})
print(f"\nStatus2: {resp2.status_code}")
print(f"Content-Type2: {resp2.headers.get('Content-Type')}")

from PIL import Image
from io import BytesIO
try:
    img2 = Image.open(BytesIO(resp2.content)).convert('RGB')
    print(f"Image2 loaded: {img2.size}")
except Exception as e:
    print(f"Image2 failed: {e}")

Status: 500
Content-Type: text/plain; charset=utf-8
Content length: 21

Status2: 200
Content-Type2: image/jpeg
Image2 loaded: (767, 432)


In [10]:
import torch
import pandas as pd
import numpy as np
import os, json

# Check exact keys
weights = torch.load(r'D:\Pics Can Lie\mmfakebench_training\aitr_mmfb_best.pt', map_location='cpu')
print("MMFakeBench AITR keys:", list(weights.keys()))

# Check what feature files exist for MMFakeBench val
val_dir = r'D:\Pics Can Lie\mmfakebench_training\val_features'
print("\nVal features:", os.listdir(val_dir))

# Check val ateeq scores
df_ateeq = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\val_ateeq_scores_full.csv')
print("\nAteeq cols:", df_ateeq.columns.tolist())
print("Ateeq rows:", len(df_ateeq))

# Check deberta val scores for MMFakeBench
df_deb = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\deberta_nli_val.csv')
print("\nDeBERTa cols:", df_deb.columns.tolist())
print("DeBERTa rows:", len(df_deb))

# Check MMFakeBench val annotations
with open(r'E:\Pics Can Lie\dataset\MMFakeBench\MMFakeBench_val.json') as f:
    val_ann = json.load(f)
print("\nMMFakeBench val samples:", len(val_ann))
print("First sample keys:", list(val_ann[0].keys()) if isinstance(val_ann, list) else list(val_ann.keys()))

# Check NewsCLIPpings test set exists
test_path = r'D:\Pics Can Lie\kaggle_dataset_full\merged_balanced\test.json'
print("\nTest set exists:", os.path.exists(test_path))
if os.path.exists(test_path):
    with open(test_path) as f:
        test_data = json.load(f)
    print("Test samples:", len(test_data))

MMFakeBench AITR keys: ['state_dict', 'epoch', 'val_metrics', 'pos_weight', 'scalar_dim']

Val features: ['clip_img.pt', 'clip_probs.npy', 'clip_sims.npy', 'clip_txt.pt', 'sample_ids.csv']

Ateeq cols: ['idx', 'sample_id', 'image_path', 'fake_cls', 'gt_answers', 'label', 'ateeq_score_ft']
Ateeq rows: 1000

DeBERTa cols: ['sample_id', 'deberta_score', 'entity_count', 'has_evidence']
DeBERTa rows: 1000

MMFakeBench val samples: 1000
First sample keys: ['text', 'image_path', 'text_source', 'image_source', 'gt_answers', 'fake_cls']

Test set exists: False


In [11]:
import torch
import numpy as np
import pandas as pd

# Get actual AITR state dict keys
checkpoint = torch.load(r'D:\Pics Can Lie\mmfakebench_training\aitr_mmfb_best.pt', map_location='cpu')
state = checkpoint['state_dict']
print("State dict keys:", list(state.keys()))
print("scalar_dim:", checkpoint['scalar_dim'])
print("epoch:", checkpoint['epoch'])
print("val_metrics:", checkpoint['val_metrics'])

# Check scalar proj shape from state dict
for k, v in state.items():
    if 'scalar' in k:
        print(f"{k}: {v.shape}")

# Check val feature shapes
img = torch.load(r'D:\Pics Can Lie\mmfakebench_training\val_features\clip_img.pt', map_location='cpu')
txt = torch.load(r'D:\Pics Can Lie\mmfakebench_training\val_features\clip_txt.pt', map_location='cpu')
probs = np.load(r'D:\Pics Can Lie\mmfakebench_training\val_features\clip_probs.npy')
sims  = np.load(r'D:\Pics Can Lie\mmfakebench_training\val_features\clip_sims.npy')
ids   = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\val_features\sample_ids.csv')
print(f"\nimg: {img.shape}, txt: {txt.shape}")
print(f"probs: {probs.shape}, sims: {sims.shape}")
print(f"ids cols: {ids.columns.tolist()}, rows: {len(ids)}")

# Check alignment between features and ateeq
df_ateeq = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\val_ateeq_scores_full.csv')
df_deb   = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\deberta_nli_val.csv')
print(f"\nAteeq sample_id sample: {df_ateeq['sample_id'].head(3).tolist()}")
print(f"DeBERTa sample_id sample: {df_deb['sample_id'].head(3).tolist()}")
print(f"IDs sample: {ids.iloc[:3, 0].tolist()}")

# Check MMFakeBench val annotation structure
import json
with open(r'E:\Pics Can Lie\dataset\MMFakeBench\MMFakeBench_val.json') as f:
    val_ann = json.load(f)
print(f"\nFirst sample: {val_ann[0]}")
print(f"fake_cls values: {set(s['fake_cls'] for s in val_ann)}")
print(f"gt_answers values: {set(s['gt_answers'] for s in val_ann)}")

State dict keys: ['cls_token', 'scalar_proj.0.weight', 'scalar_proj.0.bias', 'scalar_proj.1.weight', 'scalar_proj.1.bias', 'type_embedding.weight', 'transformer.layers.0.self_attn.in_proj_weight', 'transformer.layers.0.self_attn.in_proj_bias', 'transformer.layers.0.self_attn.out_proj.weight', 'transformer.layers.0.self_attn.out_proj.bias', 'transformer.layers.0.linear1.weight', 'transformer.layers.0.linear1.bias', 'transformer.layers.0.linear2.weight', 'transformer.layers.0.linear2.bias', 'transformer.layers.0.norm1.weight', 'transformer.layers.0.norm1.bias', 'transformer.layers.0.norm2.weight', 'transformer.layers.0.norm2.bias', 'transformer.layers.1.self_attn.in_proj_weight', 'transformer.layers.1.self_attn.in_proj_bias', 'transformer.layers.1.self_attn.out_proj.weight', 'transformer.layers.1.self_attn.out_proj.bias', 'transformer.layers.1.linear1.weight', 'transformer.layers.1.linear1.bias', 'transformer.layers.1.linear2.weight', 'transformer.layers.1.linear2.bias', 'transformer.lay

In [12]:
import numpy as np, json

with open(r'D:\Pics Can Lie\overnight_v2\task1_baseline.json') as f:
    t1 = json.load(f)
with open(r'D:\Pics Can Lie\overnight_v2\task4_ensemble.json') as f:
    t4 = json.load(f)

print("Ensembles found:", list(t4['ensembles'].keys()))
print("B+C equal:", t4['ensembles'].get('B+C_equal'))

Ensembles found: ['B+C_equal', 'B+C_Bheavy', 'B+D_equal', 'B+C+D_equal']
B+C equal: {'weights': {'B': 0.5, 'C': 0.5}, 'auc': 0.74, 'best_thr': 0.45, 'best_acc': 0.737, 'per_cat': {'original': 0.32, 'mismatch': 0.8867, 'textual_veracity_distortion': 0.9233, 'visual_veracity_distortion': 0.98}}


In [13]:
import numpy as np, pandas as pd, json

with open(r'D:\Pics Can Lie\overnight_v2\task1_baseline.json') as f:
    t1 = json.load(f)
with open(r'D:\Pics Can Lie\overnight_v2\task5_wiki_veto.json') as f:
    t5 = json.load(f)

all_probs  = np.array(t1['all_probs'])
deberta_df = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\deberta_nli_val.csv')
deberta    = deberta_df['deberta_score'].values
ids_df     = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\val_features\sample_ids.csv')
labels     = (ids_df['gt_answers'] == 'Fake').astype(int).values
fake_cls   = ids_df['fake_cls'].values

base_preds = (all_probs >= t1['best_threshold']).astype(int)

# Reproduce veto manually
veto_preds = base_preds.copy()
orig_mask  = fake_cls == 'original'
veto_cond  = orig_mask & (base_preds == 1) & (deberta >= 0.1)
veto_preds[veto_cond] = 0

print(f"Samples vetoed: {veto_cond.sum()}")
print(f"Of those, how many were actually REAL (correct veto): {((labels[veto_cond]==0)).sum()}")
print(f"Of those, how many were actually FAKE (wrong veto): {((labels[veto_cond]==1)).sum()}")
print(f"Overall with veto: {(veto_preds==labels).mean():.4f}")
print(f"Original with veto: {(veto_preds[orig_mask]==labels[orig_mask]).mean():.4f}")

Samples vetoed: 73
Of those, how many were actually REAL (correct veto): 73
Of those, how many were actually FAKE (wrong veto): 0
Overall with veto: 0.7950
Original with veto: 0.8500


In [14]:
import numpy as np, pandas as pd, json

deberta_df = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\deberta_nli_val.csv')
ids_df     = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\val_features\sample_ids.csv')
labels     = (ids_df['gt_answers'] == 'Fake').astype(int).values
fake_cls   = ids_df['fake_cls'].values
deberta    = deberta_df['deberta_score'].values

orig_mask  = fake_cls == 'original'
orig_labs  = labels[orig_mask]
orig_deb   = deberta[orig_mask]

print(f"Original samples: {orig_mask.sum()}")
print(f"Mean deberta REAL originals: {orig_deb[orig_labs==0].mean():.4f}")
print(f"Mean deberta FAKE originals: {orig_deb[orig_labs==1].mean():.4f}")
print(f"Deberta>0.1 in REAL originals: {(orig_deb[orig_labs==0]>0.1).sum()}/{(orig_labs==0).sum()}")
print(f"Deberta>0.1 in FAKE originals: {(orig_deb[orig_labs==1]>0.1).sum()}/{(orig_labs==1).sum()}")

Original samples: 300
Mean deberta REAL originals: 0.3021
Mean deberta FAKE originals: nan
Deberta>0.1 in REAL originals: 171/300
Deberta>0.1 in FAKE originals: 0/0


C:\Users\Youssef Elghandour\AppData\Local\Temp\ipykernel_2516\263321685.py:15: RuntimeWarning: Mean of empty slice.
  print(f"Mean deberta FAKE originals: {orig_deb[orig_labs==1].mean():.4f}")
d:\Pics Can Lie\venv\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [15]:
import pandas as pd
ids_df = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\val_features\sample_ids.csv')
print(ids_df.groupby(['fake_cls', 'gt_answers']).size())

fake_cls                     gt_answers
mismatch                     Fake          300
original                     True          300
textual_veracity_distortion  Fake          300
visual_veracity_distortion   Fake          100
dtype: int64


In [16]:
import numpy as np, pandas as pd, json

with open(r'D:\Pics Can Lie\overnight_v2\task1_baseline.json') as f:
    t1 = json.load(f)

deberta_df = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\deberta_nli_val.csv')
ids_df     = pd.read_csv(r'D:\Pics Can Lie\mmfakebench_training\val_features\sample_ids.csv')
labels     = (ids_df['gt_answers'] == 'Fake').astype(int).values
fake_cls   = ids_df['fake_cls'].values
deberta    = deberta_df['deberta_score'].values
all_probs  = np.array(t1['all_probs'])
base_preds = (all_probs >= t1['best_threshold']).astype(int)

orig_mask  = fake_cls == 'original'
veto_cond  = orig_mask & (base_preds == 1) & (deberta >= 0.1)
veto_preds = base_preds.copy()
veto_preds[veto_cond] = 0

# Still wrong after veto
still_wrong = orig_mask & (veto_preds == 1) & (labels == 0)
print(f"Still misclassified originals: {still_wrong.sum()}")
print(f"Their deberta scores: mean={deberta[still_wrong].mean():.4f}, "
      f"min={deberta[still_wrong].min():.4f}, max={deberta[still_wrong].max():.4f}")
print(f"Their AITR probs: mean={all_probs[still_wrong].mean():.4f}, "
      f"min={all_probs[still_wrong].min():.4f}, max={all_probs[still_wrong].max():.4f}")

# Can a higher veto threshold rescue more?
for deb_thr in [0.05, 0.08, 0.10, 0.15, 0.20]:
    vc = orig_mask & (base_preds==1) & (deberta >= deb_thr)
    vp = base_preds.copy()
    vp[vc] = 0
    acc = (vp[orig_mask] == labels[orig_mask]).mean()
    print(f"deb>{deb_thr}: vetoed={vc.sum()}, original_acc={acc:.4f}, "
          f"overall={(vp==labels).mean():.4f}")

Still misclassified originals: 45
Their deberta scores: mean=0.0102, min=0.0001, max=0.0759
Their AITR probs: mean=0.7740, min=0.6013, max=0.9465
deb>0.05: vetoed=76, original_acc=0.8600, overall=0.7980
deb>0.08: vetoed=73, original_acc=0.8500, overall=0.7950
deb>0.1: vetoed=73, original_acc=0.8500, overall=0.7950
deb>0.15: vetoed=72, original_acc=0.8467, overall=0.7940
deb>0.2: vetoed=71, original_acc=0.8433, overall=0.7930


In [21]:
import json

# Load annotations
with open(r"D:\Pics Can Lie\dataset\data\NewsClipPings\merged_balanced\test.json") as f:
    test_data = json.load(f)
annotations = test_data["annotations"]

# Load metadata for captions
with open(r"D:\Pics Can Lie\dataset\data\NewsClipPings\metadata\test.json") as f:
    meta = json.load(f)

# Check metadata structure
first_key = next(iter(meta))
print(f"Metadata keys: {list(meta.keys())[:5]}")
print(f"First entry: {meta[first_key] if not isinstance(meta, list) else meta[0]}")
print(f"Annotations count: {len(annotations)}")
print(f"Sample annotation: {annotations[0]}")

Metadata keys: ['701864', '759902', '893678', '223946', '1171296']
First entry: {'id': 701864, 'caption': 'Jim ONeill Goldman Sachs chief economist is part of a group seeking to wrest control of Manchester United from the Glazer family', 'image_path': 'visual_news/origin/guardian/images/0515/179.jpg', 'article_path': 'visual_news/origin/guardian/articles/701864.txt', 'full_article_path': 'visual_news/articles/guardian/http:::mobile-apps.guardianapis.com:items:football:2010:mar:04:jim-oneill-goldman-sachs-red-knights-200.json', 'caption_entities_spacy': [['Glazer', 'PERSON'], ['Manchester United', 'GPE'], ['Jim ONeill Goldman Sachs', 'ORG']], 'caption_entities_rel': [['Manchester_United_F.C.', 'ORG', 'Manchester United', 0.9549], ['Avram_Glazer', 'ORG', 'Glazer', 0.5603]], 'image_has_person': True, 'topic': 'football', 'source': 'guardian', 'timestamp': '2010-03-04T20:55:00Z', 'title': "Jim O'Neill faces red card from Goldman Sachs", 'title_entities_spacy': [["Jim O'Neill", 'PERSON'], [

In [25]:
import os, json, torch, clip
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

# ── paths ──────────────────────────────────────────────────────────────────
MERGED_TEST   = r"D:\Pics Can Lie\dataset\data\NewsClipPings\merged_balanced\test.json"
META_TEST     = r"D:\Pics Can Lie\dataset\data\NewsClipPings\metadata\test.json"
IMAGES_ROOT   = r"D:\Pics Can Lie\dataset\origin\origin"
CLIP_CKPT     = r"D:\Pics Can Lie\clip_finetuned_v2\clip_classifier.pt"
AITR_CKPT     = r"D:\Pics Can Lie\fusion_aitr\aitr_weights.pt"
THRESHOLD_F   = r"D:\Pics Can Lie\fusion_aitr\optimal_threshold.json"
OUT_DIR       = r"D:\Pics Can Lie\test_set_results"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── source-aware thresholds ────────────────────────────────────────────────
SOURCE_THRESHOLDS = {
    'bbc'            : 0.58,
    'washington_post': 0.57,
    'guardian'       : 0.54,
    'usa_today'      : 0.44,
}
GLOBAL_THRESHOLD = 0.54

# ── load test annotations ──────────────────────────────────────────────────
with open(MERGED_TEST) as f:
    test_data = json.load(f)
with open(META_TEST) as f:
    meta = json.load(f)

annotations = test_data["annotations"]

# Build samples: one per annotation entry (real and fake separately)
samples = []
for ann in annotations:
    mid = str(ann['id'])
    if mid not in meta:
        continue
    m = meta[mid]
    rel_path = m['image_path'].replace("visual_news/origin/", "")
    # For fake entries, image_id != id → need the swapped image path
    # Fake entries use image_id for the actual image shown
    if ann['falsified']:
        # swapped image — look up image_id in metadata
        img_mid = str(ann['image_id'])
        if img_mid not in meta:
            continue
        img_rel = meta[img_mid]['image_path'].replace("visual_news/origin/", "")
    else:
        img_rel = rel_path

    img_path = os.path.join(IMAGES_ROOT, img_rel)
    if not os.path.exists(img_path):
        continue

    samples.append({
        'id'       : ann['id'],
        'image_id' : ann['image_id'],
        'caption'  : m['caption'],
        'image_path': img_path,
        'label'    : int(ann['falsified']),   # 1=FAKE, 0=REAL
        'source'   : m.get('source', 'guardian'),
    })

print(f"Total usable samples: {len(samples)}")

# ── CLIP model ─────────────────────────────────────────────────────────────
class CLIPClassifier(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip = clip_model
        dim = 768
        self.head = nn.Sequential(
            nn.BatchNorm1d(dim * 2),
            nn.Dropout(0.5),
            nn.Linear(dim * 2, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )
    def forward(self, image, text):
        img_f = self.clip.encode_image(image).float()
        txt_f = self.clip.encode_text(text).float()
        img_f = F.normalize(img_f, dim=-1)
        txt_f = F.normalize(txt_f, dim=-1)
        x = torch.cat([img_f, txt_f], dim=-1)
        return torch.sigmoid(self.head(x)), img_f, txt_f

print("Loading CLIP v2...")
clip_base, clip_preprocess = clip.load('ViT-L/14', device=DEVICE, jit=False)
clip_base = clip_base.float()
clf = CLIPClassifier(clip_base).to(DEVICE)
ckpt = torch.load(CLIP_CKPT, map_location=DEVICE)
clf.load_state_dict(ckpt['model_state'])
clf.eval()
print("CLIP v2 loaded.")

# ── extract features ───────────────────────────────────────────────────────
class TestDataset(Dataset):
    def __init__(self, samples, preprocess):
        self.samples = samples
        self.preprocess = preprocess
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        s = self.samples[i]
        try:
            img = self.preprocess(Image.open(s['image_path']).convert('RGB'))
        except:
            img = torch.zeros(3, 224, 224)
        txt = clip.tokenize([s['caption']], truncate=True)[0]
        return img, txt, i

loader = DataLoader(TestDataset(samples, clip_preprocess),
                    batch_size=64, num_workers=0, pin_memory=False)

all_probs, all_img_feats, all_txt_feats = [], [], []

print("Extracting CLIP features...")
with torch.no_grad():
    for imgs, txts, idxs in tqdm(loader):
        imgs, txts = imgs.to(DEVICE), txts.to(DEVICE)
        probs, img_f, txt_f = clf(imgs, txts)
        all_probs.append(probs.cpu())
        all_img_feats.append(img_f.cpu())
        all_txt_feats.append(txt_f.cpu())

all_probs     = torch.cat(all_probs).squeeze()
all_img_feats = torch.cat(all_img_feats)
all_txt_feats = torch.cat(all_txt_feats)
all_sims      = (all_img_feats * all_txt_feats).sum(dim=-1)

print(f"Features extracted. probs shape: {all_probs.shape}")

# ── AITR model ─────────────────────────────────────────────────────────────
class AITRFusion(nn.Module):
    def __init__(self, feat_dim=768, scalar_dim=9, n_heads=8, n_layers=2):
        super().__init__()
        self.scalar_proj = nn.Linear(scalar_dim, feat_dim)
        self.cls_token    = nn.Parameter(torch.randn(1, 1, feat_dim))
        encoder_layer     = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=n_heads, batch_first=True, dropout=0.1)
        self.transformer  = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier   = nn.Linear(feat_dim, 1)

    def forward(self, img_f, txt_f, scalars):
        cross = img_f * txt_f
        diff  = img_f - txt_f
        sc    = self.scalar_proj(scalars)
        tokens = torch.stack([img_f, txt_f, cross, diff, sc], dim=1)
        cls = self.cls_token.expand(img_f.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        out = self.transformer(tokens)
        return torch.sigmoid(self.classifier(out[:, 0, :])).squeeze(-1)

print("Loading AITR...")
aitr = AITRFusion().to(DEVICE)
aitr.load_state_dict(torch.load(AITR_CKPT, map_location=DEVICE))
aitr.eval()
print("AITR loaded.")

# ── build scalar tensor (use zeros for missing evidence/NLI signals) ───────
# Signals: [clip_prob, clip_sim, deberta, s2, s3, s4, s5, s6, wiki_score]
# Evidence and NLI scores not available for test set → fill with val means
VAL_MEANS = {
    'deberta' : 0.05,   # approximate val mean
    's2': 0.731, 's3': 0.235, 's4': 0.620, 's5': 0.188, 's6': 0.573,
    'wiki'    : 0.05,
}

n = len(samples)
scalars = torch.zeros(n, 9)
scalars[:, 0] = all_probs
scalars[:, 1] = all_sims
scalars[:, 2] = VAL_MEANS['deberta']
scalars[:, 3] = VAL_MEANS['s2']
scalars[:, 4] = VAL_MEANS['s3']
scalars[:, 5] = VAL_MEANS['s4']
scalars[:, 6] = VAL_MEANS['s5']
scalars[:, 7] = VAL_MEANS['s6']
scalars[:, 8] = VAL_MEANS['wiki']

# ── run AITR inference ─────────────────────────────────────────────────────
print("Running AITR inference...")
all_aitr_probs = []
BS = 256
with torch.no_grad():
    for i in range(0, n, BS):
        img_b = all_img_feats[i:i+BS].to(DEVICE)
        txt_b = all_txt_feats[i:i+BS].to(DEVICE)
        sc_b  = scalars[i:i+BS].to(DEVICE)
        p = aitr(img_b, txt_b, sc_b)
        all_aitr_probs.append(p.cpu())

all_aitr_probs = torch.cat(all_aitr_probs).numpy()
labels = np.array([s['label'] for s in samples])
sources = [s['source'] for s in samples]

# ── apply source-aware thresholds ─────────────────────────────────────────
preds = np.zeros(n, dtype=int)
for i, (prob, src) in enumerate(zip(all_aitr_probs, sources)):
    thr = SOURCE_THRESHOLDS.get(src, GLOBAL_THRESHOLD)
    preds[i] = int(prob >= thr)

acc = (preds == labels).mean()
print(f"\n{'='*50}")
print(f"Test Set Accuracy (source-aware thresholds): {acc*100:.2f}%")
print(f"Total samples: {n}")

# ── per-source breakdown ───────────────────────────────────────────────────
print("\nPer-source breakdown:")
for src in ['bbc', 'guardian', 'usa_today', 'washington_post']:
    mask = np.array([s == src for s in sources])
    if mask.sum() == 0:
        continue
    src_acc = (preds[mask] == labels[mask]).mean()
    print(f"  {src:20s}: {src_acc*100:.2f}%  (n={mask.sum()})")

# ── also report global threshold result ───────────────────────────────────
preds_global = (all_aitr_probs >= GLOBAL_THRESHOLD).astype(int)
acc_global = (preds_global == labels).mean()
print(f"\nTest Set Accuracy (global threshold 0.54): {acc_global*100:.2f}%")

# ── save results ───────────────────────────────────────────────────────────
results = {
    'test_accuracy_source_aware': float(acc),
    'test_accuracy_global'      : float(acc_global),
    'n_samples'                 : n,
    'per_source'                : {}
}
for src in ['bbc', 'guardian', 'usa_today', 'washington_post']:
    mask = np.array([s == src for s in sources])
    if mask.sum() > 0:
        results['per_source'][src] = {
            'accuracy': float((preds[mask] == labels[mask]).mean()),
            'n': int(mask.sum())
        }

with open(os.path.join(OUT_DIR, 'test_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {OUT_DIR}\\test_results.json")

Total usable samples: 7264
Loading CLIP v2...


RuntimeError: Error(s) in loading state_dict for CLIPClassifier:
	Missing key(s) in state_dict: "head.0.weight", "head.0.bias", "head.0.running_mean", "head.0.running_var", "head.2.weight", "head.2.bias", "head.4.weight", "head.4.bias", "head.4.running_mean", "head.4.running_var", "head.6.weight", "head.6.bias". 
	Unexpected key(s) in state_dict: "classifier.0.weight", "classifier.0.bias", "classifier.1.weight", "classifier.1.bias", "classifier.1.running_mean", "classifier.1.running_var", "classifier.1.num_batches_tracked", "classifier.4.weight", "classifier.4.bias", "classifier.5.weight", "classifier.5.bias", "classifier.5.running_mean", "classifier.5.running_var", "classifier.5.num_batches_tracked", "classifier.8.weight", "classifier.8.bias". 

In [26]:
ckpt = torch.load(CLIP_CKPT, map_location='cpu')
for k, v in ckpt['model_state'].items():
    if 'classifier' in k:
        print(f"{k}: {v.shape}")

classifier.0.weight: torch.Size([512, 1537])
classifier.0.bias: torch.Size([512])
classifier.1.weight: torch.Size([512])
classifier.1.bias: torch.Size([512])
classifier.1.running_mean: torch.Size([512])
classifier.1.running_var: torch.Size([512])
classifier.1.num_batches_tracked: torch.Size([])
classifier.4.weight: torch.Size([128, 512])
classifier.4.bias: torch.Size([128])
classifier.5.weight: torch.Size([128])
classifier.5.bias: torch.Size([128])
classifier.5.running_mean: torch.Size([128])
classifier.5.running_var: torch.Size([128])
classifier.5.num_batches_tracked: torch.Size([])
classifier.8.weight: torch.Size([1, 128])
classifier.8.bias: torch.Size([1])
